<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap07/cap07.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di esercizi di programmazione (EP) consolida le formulazioni teoriche presentate nel Capitolo 7 — Classificazione di Immagini e Riconoscimento di Pattern — attraverso un percorso pratico applicato. Diversamente dalla manipolazione diretta dei pixel dei capitoli precedenti, gli EP di questo capitolo lavorano con le **grandezze intermedie** di un *pipeline* reale di riconoscimento di pattern — vettori di caratteristiche, distanze, etichette previste e reali, codici binari locali e istogrammi di orientamento — consentendo di validare manualmente ogni fase del ragionamento senza dipendere da librerie esterne di apprendimento automatico.

L'incatenamento degli esercizi riproduce il flusso concettuale del capitolo: si inizia con l'implementazione manuale della regola di decisione del classificatore **k-NN** su un piccolo spazio delle caratteristiche; successivamente, si rivisita, sotto l'ottica della normalizzazione delle caratteristiche, il classificatore implementato nel primo esercizio della lista; si procede con il calcolo delle metriche di **valutazione** (matrice di confusione, precisione e richiamo) a partire dalle etichette previste e reali; si continua con la codifica manuale del descrittore di tessitura **LBP** basato su un intorno $3\times3$; si approfondisce il calcolo dell'istogramma degli orientamenti del descrittore **HOG** per una singola cella; si avanza, quindi, verso l'integrazione di **estrazione di descrittori**, **classificazione k-NN** e **valutazione multi-classe** in un *pipeline* completo di riconoscimento di tessiture; e si conclude con l'applicazione di tale *pipeline* su una **immagine reale** (formato PGM), in cui il descrittore LBP viene calcolato direttamente sui pixel di un mosaico di tessiture.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella seguente:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP07_01.extensão").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare il codice Python direttamente, senza salvare un file, usa `run_code(codigo)` passando il codice come *stringa* in una variabile `codigo`:

```python
codigo = """
# ... il tuo codice qui ...
"""
TestSuite("EP07_01").run_code(codigo)
```

### 🛠️ Riepilogo dei Metodi di `morph.py` (Cap. 7)

La libreria `morph.py` offre due versioni per la maggior parte degli algoritmi: una **didattica** (metodi che terminano con `0`), implementata passo passo in NumPy, e una **classica**, basata sulle librerie `scikit-learn` e `scikit-image`. Le implementazioni didattiche sono utilizzate negli **Esercizi di Programmazione (EP)**, poiché non dipendono da librerie esterne ed eseguono entro il limite di memoria dell'ambiente **VPL** di Moodle. Le versioni classiche, invece, sono più efficienti e indicate per esperimenti in ambienti come Colab e Jupyter Notebook, ma solitamente **non possono essere utilizzate negli EP** di Moodle, poiché la libreria `scikit-learn` supera la memoria disponibile nel VPL.

1. **Lettura dei dati (`readClasses`, `readDataset`, `readTrain`, `readTest`)**  
   Standardizzano l'input dei set di addestramento e test, restituendo le matrici delle caratteristiche ($X$) e i vettori delle etichette ($y$).

2. **Classificazione (`knn0` / `knn`)**  
   Implementano l'algoritmo dei **k-vicini più prossimi (k-NN)** per la classificazione binaria e multiclasse, utilizzando la distanza Euclidea o di Manhattan.

3. **Normalizzazione (`zscore0` / `zscore`)**  
   Applicano la normalizzazione *z-score* agli attributi, riducendo le differenze di scala prima della classificazione.

4. **Valutazione (`confusion0` / `confusion`)**  
   Calcolano la matrice di confusione e metriche come accuratezza, precisione e richiamo, sia per problemi binari che multiclasse.

5. **Descrittore di texture (`lbp0` / `lbp`)**  
   Calcolano il ***Local Binary Pattern* (LBP)**, consentendo di ottenere la mappa LBP, il codice di un pixel o l'istogramma di una regione dell'immagine.

6. **Descrittore di forma (`hog0` / `hog`)**  
   Calcolano l'***Histogram of Oriented Gradients* (HOG)**, producendo istogrammi delle orientazioni dei gradienti per rappresentare informazioni di forma e contorno.

### EP07_01 🟢 Classificatore k-NN Passo dopo Passo

Il `KNeighborsClassifier` di `scikit-learn`, utilizzato per tutto il capitolo, nasconde dietro una singola chiamata (`.fit` / `.predict`) una regola decisionale piuttosto semplice: per ogni nuova osservazione, calcolare la distanza da tutti gli esempi di addestramento, selezionare i $k$ più vicini e votare per la classe maggioritaria tra di essi.

Prima di affidarsi alla libreria, ti è stato chiesto di implementare questa regola da zero, per uno spazio delle caratteristiche bidimensionale, esattamente come fa internamente il simulatore interattivo della frontiera di decisione del capitolo a ogni clic dell’utente.

#### 📋 Linee guida di implementazione

1. **Quantità e parametro:** leggere l’intero $N$ (numero di esempi di addestramento) e l’intero dispari $k$ (numero di vicini).
2. **Esempi di addestramento:** per ciascuno degli $N$ esempi, leggere tre valori: le coordinate $x$ e $y$ (reali) e l’etichetta $r$ (intero, $0$ o $1$).
3. **Query:** leggere l’intero $Q$ (numero di punti di query) e, successivamente, le coordinate $x_q$, $y_q$ (reali) di ciascuna query.
4. **Distanza:** per ogni query, calcolare la distanza euclidea verso **tutti** gli esempi di addestramento:
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Selezione dei vicini:** ordinare gli esempi per distanza crescente e selezionare i primi $k$. In caso di **pareggio di distanza** al confine del k-esimo vicino, risolvere a favore dell’esempio letto **per primo** in input (ordine di lettura stabile).
6. **Votazione maggioritaria:** contare i voti di ciascuna classe tra i $k$ vicini selezionati. In caso di **pareggio nella votazione** (possibile solo se $k$ è pari, cosa che non dovrebbe verificarsi secondo la linea guida del punto 1, ma da gestire in modo difensivo), assegnare la classe del vicino più vicino tra le classi in pareggio.
7. **Output:** per ogni query, nell’ordine di input, stampare la classe prevista. Alla fine, stampare il totale delle query classificate come classe `1`.

#### 📌 Vincoli Computazionali

* **Metrica fissa:** utilizzare esclusivamente la distanza euclidea (non la *distanza al quadrato*) per l’ordinamento, sebbene il risultato del confronto sia lo stesso.
* **k sempre dispari:** l’input garantisce $k$ dispari e $k \le N$; ciononostante, implementare il pareggio del punto 6 per robustezza.
* **Stabilità:** nell’ordinare per distanza, preservare l’ordine relativo degli esempi con la stessa distanza (ordinamento stabile).

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nel k-NN |
|---|---|
| Spazio delle caratteristiche | Insieme di tutti i vettori $(x, y)$ possibili |
| Distanza euclidea | Misura di similarità tra osservazioni |
| $k$ piccolo | Frontiera irregolare, alta varianza |
| $k$ grande | Frontiera liscia, alto bias |
| Votazione maggioritaria | Regola decisionale $\hat y = \operatorname{moda}\{y_i : x_i \in N_k(x)\}$ |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $N$ e $k$, separati da spazio.
* Prossime $N$ righe: tre valori per riga — $x$, $y$ (reali) e $r$ (intero $\in \{0,1\}$), separati da spazio.
* Riga successiva: intero $Q$.
* Prossime $Q$ righe: due valori per riga — $x_q$, $y_q$ (reali), separati da spazio.

**Output:**

* $Q$ righe, ciascuna con la classe prevista (`0` o `1`) per la rispettiva query, nell’ordine di input.
* Ultima riga: `Totale classe 1: X`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Totale classe 1: 0 | Query vicina al gruppo di classe 0. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Totale classe 1: 1 | Con $k=1$, ogni query eredita la classe del vicino più prossimo. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0701" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0701 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0701 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0701 button:hover { background: #e8dfcf; }
  #sim-ep0701 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0701_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0701_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_01: Classificatore k-NN Passo per Passo</span>
  <span class="sim-ep0701_pill">Votazione a maggioranza</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0701_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Numero di vicini (k): <span id="sim-ep0701_vl" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    
    <input id="sim-ep0701_sl" type="range" min="1" max="7" step="2" value="3">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola k e osserva quali esempi di addestramento (ordinati per distanza) partecipano alla votazione per la query fissa (&starf; in x = 3, y = 3).
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0701_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0701_debug" class="sim-ep0701_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep01(root){
    if (!root || root.dataset.sim07Ep01Init) return;
    root.dataset.sim07Ep01Init = "1";

    var query = {x: 3, y: 3};
    var pontos = [
      {nome: "A", x: 0, y: 0, r: 0},
      {nome: "B", x: 1, y: 0, r: 0},
      {nome: "C", x: 5, y: 5, r: 1},
      {nome: "D", x: 6, y: 5, r: 1},
      {nome: "E", x: 2, y: 2, r: 0},
      {nome: "F", x: 4, y: 4, r: 1},
      {nome: "G", x: 0, y: 2, r: 0},
      {nome: "H", x: 6, y: 3, r: 1}
    ];

    pontos.forEach(function(p, i){
      p.d = Math.sqrt(Math.pow(p.x - query.x, 2) + Math.pow(p.y - query.y, 2));
      p.idx = i;
    });

    pontos.sort(function(a, b){
      return (a.d - b.d) || (a.idx - b.idx);
    });

    var slEl  = root.querySelector('#sim-ep0701_sl');
    var vlEl  = root.querySelector('#sim-ep0701_vl');
    var cards = root.querySelector('#sim-ep0701_cards');
    var dbg   = root.querySelector('#sim-ep0701_debug');

    function render(){
      var k = parseInt(slEl.value, 10);
      vlEl.textContent = k;
      cards.innerHTML = '';
      var votos = [0, 0];

      pontos.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">d = ' + p.d.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : pontos[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] + '  |  Classe prevista: ' + previsto;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim07Ep01(){
    var root = document.getElementById('sim-ep0701');
    if (root) initSim07Ep01(root); else setTimeout(tryInitSim07Ep01, 200);
  }
  tryInitSim07Ep01();
})();
</script>
""")

**Figura 7.1:** Simulatore EP07_01: Classificatore k-NN Passo dopo Passo


<figure id="fig-07-sim-ep0701">
  <img src="imagens/fig-07-sim-ep0701.png" alt=" Simulatore EP07_01: Classificatore k-NN Passo dopo Passo " style="max-width:80%" />
  <figcaption><strong>Figura 7.1:</strong>  Simulatore EP07_01: Classificatore k-NN Passo dopo Passo </figcaption>
</figure>

In [ ]:
%%writefile EP07_01.py
# Codice Python

In [ ]:
TestSuite("EP07_01.py").run()

### EP07_02 🟡 Normalizzazione *Z-score* e Robustezza del k-NN a Scale Distinte

Questo esercizio riprende il classificatore implementato nell'**EP07_01**, questa volta sotto la prospettiva discussa nella sezione *L'Impatto della Scala e la Normalizzazione delle Caratteristiche* del capitolo: il k-NN decide in base alla distanza tra vettori, per cui una caratteristica misurata su una scala molto più ampia rispetto alle altre tende a **dominare** il calcolo della distanza, anche quando non è la più rilevante per separare le classi.

Un sistema di ispezione registra, per ogni pezzo, la sua **area** (in pixel, che può raggiungere centinaia o migliaia) e la sua **circolarità** (sempre compresa tra $0$ e $1$). Ti è stato affidato il compito di classificare nuovi pezzi con il k-NN in due modi — con e senza la standardizzazione *Z-score* presentata nel capitolo — e di riportare in quali casi i due approcci **divergono**.

#### 📋 Linee Guida di Implementazione

1. **Quantità e parametro:** Leggere l'intero $N$ (numero di esempi di addestramento) e l'intero dispari $k$.
2. **Esempi di addestramento:** Per ciascuno degli $N$ esempi, leggere tre valori: l'area $x_1$ (reale), la circolarità $x_2$ (reale) e l'etichetta $r$ (intero, $0$ o $1$).
3. **Query:** Leggere l'intero $Q$ e, successivamente, le coordinate $x_1, x_2$ di ciascuna query.
4. **Classificazione senza normalizzazione:** Per ogni query, classificarla con il k-NN direttamente su $(x_1, x_2)$, con distanza euclidea e le stesse regole di spareggio dell'EP07_01 (ordine di lettura per distanze a pari merito; vicino più prossimo tra classi a pari merito nella votazione).
5. **Parametri di normalizzazione:** Calcolare la media $\mu_j$ e la deviazione standard **popolazionale** $\sigma_j$ (divisione per $N$, non per $N-1$ — la stessa convenzione adottata dalla classe `StandardScaler`) di ogni caratteristica $j \in \{1,2\}$, **esclusivamente sul set di addestramento**.
6. **Standardizzazione:** Trasformare ogni caratteristica di addestramento e di query tramite
$$
z_j = \frac{x_j - \mu_j}{\sigma_j}.
$$
Se $\sigma_j = 0$ (caratteristica costante nell'addestramento), definire $z_j = 0$ per tutti i campioni di quella caratteristica, evitando la divisione per zero.
7. **Classificazione con normalizzazione:** Ripetere la classificazione k-NN del punto 4, ora sui vettori standardizzati $(z_1, z_2)$, con le stesse regole di spareggio.
8. **Output:** Per ogni query, nell'ordine di input, stampare le due classi previste. Alla fine, stampare il numero di query in cui le due classificazioni **divergono**.

#### 📌 Vincoli Computazionali

* **Adattamento solo sul training set:** $\mu_j$ e $\sigma_j$ sono calcolati unicamente a partire dal set di addestramento e riapplicati alle query — mai ricalcolati a partire da esse. Questa pratica evita la **dispersione dei dati** (*data leakage*), menzionata nella sezione sulla normalizzazione del capitolo.
* **Deviazione standard popolazionale:** utilizzare $\sigma_j = \sqrt{\frac{1}{N}\sum_i (x_{i,j}-\mu_j)^2}$, e non la versione campionaria (divisione per $N-1$).
* **Caratteristica costante:** trattare $\sigma_j = 0$ come caso speciale (punto 6); non deve verificarsi un errore di divisione per zero.
* **Regole di spareggio:** riutilizzare esattamente le convenzioni dell'EP07_01, sia nella selezione dei $k$ vicini sia nella votazione a maggioranza.

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo |
|---|---|
| Standardizzazione *Z-score* | Riscalare ogni caratteristica per media $0$ e deviazione standard $1$, rendendo scale eterogenee confrontabili |
| Adattamento (*fit*) solo sul training set | Garantire che la valutazione sulle query rifletta solo ciò che il modello ha appreso nell'addestramento |
| Distanza euclidea senza normalizzazione | Dominata dalla caratteristica con maggiore ampiezza — qui, l'area |
| Previsione divergente | Evidenzia che la scala delle caratteristiche, e non solo l'algoritmo o i dati, può determinare il confine decisionale del k-NN |

Questo esercizio rafforza, in modo controllato, il motivo per cui il `StandardScaler` viene applicato prima del k-NN lungo il capitolo: senza questa fase, le caratteristiche di circolarità — anche se altamente discriminative — possono essere praticamente ignorate dal classificatore di fronte a una caratteristica di area con ampiezza centinaia di volte maggiore.

#### 📦 Specifiche di Input e Output (VPL)

**Input:**

* Riga 1: Interi $N$ e $k$, separati da spazio.
* Successive $N$ righe: tre valori per riga — $x_1$, $x_2$ (reali) e $r$ (intero $\in \{0,1\}$), separati da spazio.
* Riga successiva: intero $Q$.
* Successive $Q$ righe: due valori per riga — $x_1$, $x_2$ (reali) della query, separati da spazio.

**Output:**

* $Q$ righe, nel formato `SemNorm=<0|1> ComNorm=<0|1>`, nell'ordine di input delle query.
* Ultima riga: `Divergiu: <int>`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 3<br>10 0.9 0<br>12 0.85 0<br>900 0.2 1<br>950 0.25 1<br>1<br>500 0.88 | SemNorm=1 ComNorm=0<br>Divergiu: 1 | Senza normalizzazione, l'area (scala di centinaia) domina la distanza e la query viene classificata come classe `1`. Dopo la standardizzazione, la circolarità — molto più vicina ai campioni di classe `0` — inizia a pesare in modo comparabile, e la previsione cambia a `0`. |
| 2 1<br>0 0.5 0<br>100 0.5 1<br>1<br>60 0.5 | SemNorm=1 ComNorm=1<br>Divergiu: 0 | La circolarità è costante nell'addestramento ($\sigma_2=0$); secondo la regola del punto 6, $z_2=0$ per tutti i campioni, e la classificazione dipende solo dall'area in entrambi i casi. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0702" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0702 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0702 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0702 button:hover { background: #e8dfcf; }
  #sim-ep0702 button.sim-ep0702_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0702_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0702_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_02: Normalizzazione Z-score e Distanza k-NN</span>
  <span class="sim-ep0702_pill">Standardizzazione delle Caratteristiche</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Seleção de Modo -->
  <div class="sim-ep0702_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:8px;">
      Ogni esempio ha due caratteristiche: area (px) e circolarità [0, 1]. Alterna la normalizzazione e osserva il cambiamento nella classe prevista.
    </div>
    
    <div id="sim-ep0702_query" style="font-size:11px; color:#26241d; text-align:center; font-family:monospace; font-weight:700; margin-bottom:10px;"></div>

    <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
      <button id="sim-ep0702_btn_raw" class="sim-ep0702_active">Senza Normalizzazione</button>
      <button id="sim-ep0702_btn_norm">Con Normalizzazione (Z-score)</button>
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0702_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0702_debug" class="sim05_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep02(root){
    if (!root || root.dataset.sim07Ep02Init) return;
    root.dataset.sim07Ep02Init = "1";

    var pontos = [
      {nome: "P1", x1: 10,  x2: 0.90, r: 0},
      {nome: "P2", x1: 12,  x2: 0.85, r: 0},
      {nome: "P3", x1: 900, x2: 0.20, r: 1},
      {nome: "P4", x1: 950, x2: 0.25, r: 1}
    ];

    pontos.forEach(function(p, i){ p.idx = i; });
    var query = {x1: 500, x2: 0.88};
    var k = 3;

    function stats(vals){
      var m = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
      var v = vals.reduce(function(a, s){ return a + (s - m) * (s - m); }, 0) / vals.length;
      return {mean: m, std: Math.sqrt(v)};
    }

    var s1 = stats(pontos.map(function(p){ return p.x1; }));
    var s2 = stats(pontos.map(function(p){ return p.x2; }));

    function z(x, s){ return s.std === 0 ? 0 : (x - s.mean) / s.std; }

    var cards   = root.querySelector('#sim-ep0702_cards');
    var dbg     = root.querySelector('#sim-ep0702_debug');
    var qEl     = root.querySelector('#sim-ep0702_query');
    var btnRaw  = root.querySelector('#sim-ep0702_btn_raw');
    var btnNorm = root.querySelector('#sim-ep0702_btn_norm');
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle('sim-ep0702_active', !modoNorm);
      btnNorm.classList.toggle('sim-ep0702_active', modoNorm);

      qEl.textContent = '★ Query: Area = ' + query.x1 + ', Circularidade = ' + query.x2 +
        (modoNorm ? ' → z_área = ' + z(query.x1, s1).toFixed(3) + ', z_circ = ' + z(query.x2, s2).toFixed(3) : '');

      var qx1 = modoNorm ? z(query.x1, s1) : query.x1;
      var qx2 = modoNorm ? z(query.x2, s2) : query.x2;

      var lista = pontos.map(function(p){
        var px1 = modoNorm ? z(p.x1, s1) : p.x1;
        var px2 = modoNorm ? z(p.x2, s2) : p.x2;
        var d = Math.sqrt((px1 - qx1) * (px1 - qx1) + (px2 - qx2) * (px2 - qx2));
        return {nome: p.nome, r: p.r, d: d, idx: p.idx, area: p.x1, circ: p.x2, va: px1, vc: px2};
      });

      lista.sort(function(a, b){ return (a.d - b.d) || (a.idx - b.idx); });

      cards.innerHTML = '';
      var votos = [0, 0];

      lista.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        var valorUsado = modoNorm
          ? ('z = (' + p.va.toFixed(2) + ', ' + p.vc.toFixed(2) + ')')
          : ('área = ' + p.area + ', circ = ' + p.circ);

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">' + valorUsado + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px;">d = ' + p.d.toFixed(3) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : lista[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = (modoNorm ? 'COM Normalização' : 'SEM Normalização') +
        '  |  k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] +
        '  |  Classe prevista: ' + previsto;
    }

    btnRaw.addEventListener('click', function(){ modoNorm = false; render(); });
    btnNorm.addEventListener('click', function(){ modoNorm = true; render(); });
    render();
  }

  function tryInitSim07Ep02(){
    var root = document.getElementById('sim-ep0702');
    if (root) initSim07Ep02(root); else setTimeout(tryInitSim07Ep02, 200);
  }
  tryInitSim07Ep02();
})();
</script>
""")

**Figura 7.2:** Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN


<figure id="fig-07-sim-ep0702">
  <img src="imagens/fig-07-sim-ep0702.png" alt=" Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN " style="max-width:80%" />
  <figcaption><strong>Figura 7.2:</strong>  Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN </figcaption>
</figure>

In [ ]:
%%writefile EP07_02.py
# Codice Python

In [ ]:
TestSuite("EP07_02.py").run()

### EP07_03 🟡 Valutazione tramite Matrice di Confusione

Un classificatore binario per la qualità della saldatura è stato addestrato e testato su una linea di produzione. Per ogni pezzo ispezionato, il sistema ha registrato l'etichetta **reale** (ottenuta da un esperto) e l'etichetta **prevista** dal classificatore, dove `1` rappresenta "difettoso" e `0` rappresenta "conforme".

La direzione qualità vuole conoscere non solo l'accuratezza del sistema, ma anche la sua **precisione** (quando il sistema segnala un difetto, con che frequenza ha ragione?) e il suo **richiamo** (di tutti i pezzi realmente difettosi, quanti il sistema è riuscito a identificare?) — la distinzione discussa nella sezione sulla valutazione dei classificatori del capitolo.

#### 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere il numero intero $N$ (numero di pezzi ispezionati).
2. **Dati di ciascun pezzo:** Per ciascuno degli $N$ pezzi, leggere due interi — l'etichetta reale $y$ e l'etichetta prevista $\hat y$ (entrambe $\in \{0, 1\}$).
3. **Matrice di confusione:** Considerando la classe `1` (difettoso) come **positiva**, contare:
   - $VP$ (Vero Positivo): $y=1$ e $\hat y=1$;
   - $FP$ (Falso Positivo): $y=0$ e $\hat y=1$;
   - $FN$ (Falso Negativo): $y=1$ e $\hat y=0$;
   - $VN$ (Vero Negativo): $y=0$ e $\hat y=0$.
4. **Metriche:** Calcolare
$$
\text{Accuratezza} = \frac{VP+VN}{N}, \quad
\text{Precisione} = \frac{VP}{VP+FP}, \quad
\text{Richiamo} = \frac{VP}{VP+FN}.
$$
5. **Casi degenerati:** Se $VP+FP=0$ (nessuna predizione positiva), stampare `Precisao: indefinida`. Se $VP+FN=0$ (nessun caso positivo reale), stampare `Revocacao: indefinida`.
6. **Arrotondamento:** Tutte le metriche numeriche devono essere arrotondate a 4 cifre decimali (*round half away from zero*) solo nella visualizzazione.

#### 📌 Vincoli Computazionali

* **Convenzione di classe positiva fissa:** la classe `1` è sempre la classe positiva in questo esercizio, indipendentemente dalla sua frequenza relativa.
* **Protezione dalla divisione per zero:** implementare i casi degenerati del punto 5 prima di eseguire la divisione.
* **Ordine di output:** seguire esattamente l'ordine specificato nella sezione di output, anche nei casi degenerati.

#### 🧠 Fondamento Teorico

| Metrica | Domanda a cui risponde | Sensibile allo sbilanciamento? |
|---|---|---|
| Accuratezza | Quale frazione di pezzi è stata classificata correttamente? | Sì — può mascherare errori nella classe minoritaria |
| Precisione | Dei pezzi segnalati come difettosi, quanti lo sono realmente? | Penalizza i falsi positivi |
| Richiamo | Dei pezzi realmente difettosi, quanti sono stati rilevati? | Penalizza i falsi negativi |

In un contesto industriale, un **richiamo** basso è spesso più grave di una **precisione** bassa: lasciar passare un pezzo difettoso (falso negativo) tende a essere più costoso che ispezionare manualmente un pezzo buono segnalato per errore (falso positivo).

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Numero intero $N$.
* Prossime $N$ righe: due interi per riga — $y$ e $\hat y$, separati da spazio.

**Output (in questo ordine esatto):**

```
VP=<int> FP=<int> FN=<int> VN=<int>
Acuracia: <valore o metrica indefinita>
Precisao: <valore o indefinita>
Revocacao: <valore o indefinita>
```

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>1 1<br>0 1<br>1 0<br>0 0 | VP=1 FP=1 FN=1 VN=1<br>Acuracia: 0.5000<br>Precisao: 0.5000<br>Revocacao: 0.5000 | Un errore di ciascun tipo. |
| 3<br>0 0<br>0 0<br>0 0 | VP=0 FP=0 FN=0 VN=3<br>Acuracia: 1.0000<br>Precisao: indefinida<br>Revocacao: indefinida | Nessun caso positivo reale né previsto. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0703" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0703 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0703 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0703 button:hover { background: #e8dfcf; }
  #sim-ep0703 button.sim-ep0703_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0703_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0703_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_03: Precisione x Richiamo</span>
  <span class="sim-ep0703_pill">Linea di produzione</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Cenário -->
  <div class="sim-ep0703_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Scegli uno scenario di ispezione e osserva come Accuratezza, Precisione e Richiamo reagiscono in modo diverso.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0703_b1" class="sim-ep0703_active">Scenario A: Errori bilanciati</button>
      <button id="sim-ep0703_b2">Scenario B: Falsi negativi</button>
      <button id="sim-ep0703_b3">Scenario C: Nessun difetto reale</button>
      <button id="sim-ep0703_b4">Scenario D: Falsi positivi</button>
    </div>
  </div>

  <!-- Cards das Peças do Cenário -->
  <div id="sim-ep0703_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0703_debug" class="sim-ep0703_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep03(root){
    if (!root || root.dataset.sim07Ep03Init) return;
    root.dataset.sim07Ep03Init = "1";

    var cenarios = {
      A: [{y:1, p:1}, {y:0, p:1}, {y:1, p:0}, {y:0, p:0}],
      B: [{y:1, p:0}, {y:1, p:0}, {y:1, p:1}, {y:0, p:0}],
      C: [{y:0, p:0}, {y:0, p:0}, {y:0, p:0}],
      D: [{y:0, p:1}, {y:0, p:1}, {y:1, p:1}, {y:0, p:0}]
    };

    var cards = root.querySelector('#sim-ep0703_cards');
    var dbg   = root.querySelector('#sim-ep0703_debug');

    var botoes = {
      A: root.querySelector('#sim-ep0703_b1'),
      B: root.querySelector('#sim-ep0703_b2'),
      C: root.querySelector('#sim-ep0703_b3'),
      D: root.querySelector('#sim-ep0703_b4')
    };

    function render(key){
      Object.keys(botoes).forEach(function(k){
        botoes[k].classList.toggle('sim-ep0703_active', k === key);
      });

      var dados = cenarios[key];
      var VP = 0, FP = 0, FN = 0, VN = 0;
      cards.innerHTML = '';

      dados.forEach(function(d, i){
        if (d.y === 1 && d.p === 1) VP++;
        else if (d.y === 0 && d.p === 1) FP++;
        else if (d.y === 1 && d.p === 0) FN++;
        else VN++;

        var statusCor = '';
        var statusTxt = '';

        if (d.y === d.p) {
          statusCor = 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
          statusTxt = d.y === 1 ? 'VP (Acerto)' : 'VN (Acerto)';
        } else {
          statusCor = 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;';
          statusTxt = d.p === 1 ? 'FP (Alarme Falso)' : 'FN (Escapou)';
        }

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' + statusCor;

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">Peça ' + (i + 1) + '</div>' +
          '<div style="font-family:monospace; font-size:10px; margin-bottom:4px;">Real = ' + d.y + ' | Prev = ' + d.p + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + statusTxt + '</div>';

        cards.appendChild(div);
      });

      var N = dados.length;
      var acc = ((VP + VN) / N).toFixed(4);
      var prec = (VP + FP) > 0 ? (VP / (VP + FP)).toFixed(4) : 'Indefinida';
      var rev = (VP + FN) > 0 ? (VP / (VP + FN)).toFixed(4) : 'Indefinida';

      if (prec === 'Indefinida' || parseFloat(prec) < 0.5) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'VP = ' + VP + ' | FP = ' + FP + ' | FN = ' + FN + ' | VN = ' + VN +
        '  |  Acurácia = ' + acc + '  |  Precisão = ' + prec + '  |  Revocação = ' + rev;
    }

    botoes.A.addEventListener('click', function(){ render('A'); });
    botoes.B.addEventListener('click', function(){ render('B'); });
    botoes.C.addEventListener('click', function(){ render('C'); });
    botoes.D.addEventListener('click', function(){ render('D'); });

    render('A');
  }

  function tryInitSim07Ep03(){
    var root = document.getElementById('sim-ep0703');
    if (root) initSim07Ep03(root); else setTimeout(tryInitSim07Ep03, 200);
  }
  tryInitSim07Ep03();
})();
</script>
""")

**Figura 7.3:** Simulatore EP07_03: Precisione x Richiamo


<figure id="fig-07-sim-ep0703">
  <img src="imagens/fig-07-sim-ep0703.png" alt=" Simulatore EP07_03: Precisione x Richiamo " style="max-width:80%" />
  <figcaption><strong>Figura 7.3:</strong>  Simulatore EP07_03: Precisione x Richiamo </figcaption>
</figure>

In [ ]:
%%writefile EP07_03.py
# Codice Python

In [ ]:
TestSuite("EP07_03.py").run()

### EP07_04 🟠 Codifica Manuale del Descrittore LBP

La funzione `local_binary_pattern` di `scikit-image`, utilizzata nel progetto di classificazione delle trame, calcola automaticamente il codice LBP di ciascun pixel di un'immagine. Prima di usarla come una scatola nera, ti è stato affidato il compito di implementare manualmente il calcolo del codice LBP classico ($P=8$, $R=1$) per il pixel centrale di un intorno $3\times3$, esattamente come definito nell'equazione del capitolo.

Oltre al codice, il sistema di ispezione delle trame deve anche sapere se quel pattern è **uniforme** — un pattern è uniforme quando il numero di transizioni ($0\to1$ o $1\to0$) percorrendo gli 8 bit **circolarmente** (tornando dall'ultimo bit al primo) è **al massimo 2**, proprietà sfruttata dalla variante *uniforme* del LBP menzionata nel capitolo.

#### 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere l'intero $T$ (numero di intorni da elaborare).
2. **Dati di ciascun intorno:** Per ciascuno dei $T$ intorni, leggere una matrice $3\times3$ di interi (intensità), fornita in 3 righe di 3 valori ciascuna. Il pixel centrale è la posizione `[1][1]`.
3. **Ordine dei vicini:** Percorrere gli 8 vicini in senso **orario**, iniziando dall'angolo superiore sinistro, nel seguente ordine di posizioni `[riga][colonna]`: `[0][0]`, `[0][1]`, `[0][2]`, `[1][2]`, `[2][2]`, `[2][1]`, `[2][0]`, `[1][0]`. Questo è l'indice $p = 0, 1, \ldots, 7$ dell'equazione del LBP.
4. **Funzione soglia:** Per ciascun vicino $p$ con intensità $g_p$ e centro $g_c$, calcolare $s(g_p - g_c)$, che vale `1` se $g_p \geq g_c$ e `0` altrimenti.
5. **Codice LBP:** Calcolare
$$
\mathrm{LBP} = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p.
$$
6. **Transizioni:** Considerando la sequenza circolare di bit $s_0, s_1, \ldots, s_7$ (nell'ordine del punto 3), contare quante coppie consecutive **adiacenti nella sequenza circolare** (inclusa la coppia $s_7, s_0$) differiscono tra loro.
7. **Classificazione:** Se il numero di transizioni è $\le 2$, classificare come `UNIFORME`; altrimenti, `NAO_UNIFORME`.
8. **Output:** Per ciascun intorno, nell'ordine di ingresso, stampare il codice LBP (intero decimale, $0$–$255$), il numero di transizioni e la classificazione.

#### 📌 Vincoli Computazionali

* **Ordine fisso dei vicini:** l'ordine del punto 3 è obbligatorio — invertirlo produce un codice numericamente diverso, anche se rappresenta lo stesso pattern visivo.
* **Confronto non stretto:** $s(z) = 1$ quando $z \ge 0$ (il capitolo stesso definisce l'uguaglianza come inclusa nel caso `1`).
* **Conteggio circolare:** non dimenticare la coppia che chiude il ciclo ($s_7$ con $s_0$); ignorare questa coppia è un errore comune che classifica erroneamente i pattern uniformi.

#### 🧠 Fondamento Teorico

| Pattern (bit $s_0\ldots s_7$) | Transizioni | Interpretazione |
|---|---|---|
| `00000000` o `11111111` | 0 | Regione omogenea (macchia chiara o scura) |
| `00001111` | 2 | Bordo semplice tra due regioni |
| `01010101` | 8 | Trama a contrasto alternato — non uniforme |

I pattern uniformi si concentrano in regioni di trama uniforme o bordi semplici; i pattern non uniformi tendono a corrispondere a rumore ad alta frequenza. Per questo, l'istogramma LBP *uniforme*, usato nel progetto di classificazione delle trame, raggruppa tutti i pattern non uniformi in un unico contenitore, riducendo la dimensionalità del descrittore.

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $T$.
* Per ciascun intorno: 3 righe con 3 interi ciascuna (matrice $3\times3$).

**Uscita:**

* $T$ righe, nel formato `LBP=<int> transicoes=<int> <UNIFORME|NAO_UNIFORME>`.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 1<br>10 10 10<br>10 50 10<br>10 10 10 | LBP=0 transicoes=0 UNIFORME | Il centro è il più chiaro; tutti i vicini generano bit 0. |
| 1<br>90 90 90<br>10 50 10<br>90 90 90 | LBP=119 transicoes=4 NAO_UNIFORME | Vicini chiari e scuri alternati nell'intorno. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0704" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0704 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0704 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0704 button:hover { background: #e8dfcf; }
  #sim-ep0704 button.sim-ep0704_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0704_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0704_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_04: Codice LBP di un Intorno 3&times;3</span>
  <span class="sim-ep0704_pill">P = 8, R = 1</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Exemplo -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Clicca su una cella dell'intorno per alternare tra chiaro e scuro (il centro è fisso) e osserva il codice LBP risultante. L'etichetta p indica l'indice dell'equazione.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0704_b1">Esempio 1: Macchia Omogenea</button>
      <button id="sim-ep0704_b2" class="sim-ep0704_active">Esempio 2: Pattern Alternato</button>
    </div>
  </div>

  <!-- Grid Vizinhança 3x3 -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px; text-align:center;">
    <div id="sim-ep0704_grid" style="display:grid; grid-template-columns:repeat(3, 60px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0704_debug" class="sim-ep0704_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep04(root){
    if (!root || root.dataset.sim07Ep04Init) return;
    root.dataset.sim07Ep04Init = "1";

    var exemplos = {
      1: [[10, 10, 10], [10, 50, 10], [10, 10, 10]],
      2: [[90, 90, 90], [10, 50, 10], [90, 90, 90]]
    };

    var valores = exemplos[2].map(function(row){ return row.slice(); });
    var grid = root.querySelector('#sim-ep0704_grid');
    var dbg  = root.querySelector('#sim-ep0704_debug');
    var btn1 = root.querySelector('#sim-ep0704_b1');
    var btn2 = root.querySelector('#sim-ep0704_b2');

    var ordem = [[0, 0], [0, 1], [0, 2], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0]];
    var pIndex = {};
    ordem.forEach(function(pos, p){ pIndex[pos[0] + ',' + pos[1]] = p; });
    var cenarioAtivo = 2;

    function marcarBotaoAtivo(n){
      cenarioAtivo = n;
      btn1.classList.toggle('sim-ep0704_active', n === 1);
      btn2.classList.toggle('sim-ep0704_active', n === 2);
    }

    function render(){
      grid.innerHTML = '';
      for (var r = 0; r < 3; r++){
        for (var c = 0; c < 3; c++){
          (function(r, c){
            var v = valores[r][c];
            var central = (r === 1 && c === 1);
            var div = document.createElement('div');

            var bordaCor = central ? '#26241d' : '#e4dcc8';
            var textoCor = v > 128 ? '#26241d' : '#ffffff';

            div.style.cssText = 'position:relative; height:60px; display:flex; align-items:center; justify-content:center; font-family:monospace; font-weight:700; border-radius:6px; cursor:' + (central ? 'default' : 'pointer') + '; border:2px solid ' + bordaCor + '; background:rgb(' + v + ',' + v + ',' + v + '); color:' + textoCor + '; transition:all 0.15s ease;';
            div.textContent = v;

            if (!central){
              var pLabel = document.createElement('span');
              pLabel.textContent = 'p' + pIndex[r + ',' + c];
              pLabel.style.cssText = 'position:absolute; top:2px; left:4px; font-size:9px; font-weight:400; opacity:0.8;';
              div.appendChild(pLabel);

              div.addEventListener('click', function(){
                valores[r][c] = valores[r][c] >= 128 ? 10 : 200;
                cenarioAtivo = null;
                btn1.classList.remove('sim-ep0704_active');
                btn2.classList.remove('sim-ep0704_active');
                render();
              });
            }
            grid.appendChild(div);
          })(r, c);
        }
      }

      var gc = valores[1][1];
      var bits = ordem.map(function(pos){ return valores[pos[0]][pos[1]] >= gc ? 1 : 0; });
      var lbp = 0;
      bits.forEach(function(b, p){ lbp += b * Math.pow(2, p); });

      var trans = 0;
      for (var i = 0; i < 8; i++){
        if (bits[i] !== bits[(i + 1) % 8]) trans++;
      }

      var classe = trans <= 2 ? 'UNIFORME' : 'NÃO-UNIFORME';

      if (trans <= 2) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'bit (p0..p7) = ' + bits.join('') + '  |  LBP = ' + lbp + '  |  transições = ' + trans + '  |  ' + classe;
    }

    btn1.addEventListener('click', function(){
      valores = exemplos[1].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(1);
      render();
    });

    btn2.addEventListener('click', function(){
      valores = exemplos[2].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(2);
      render();
    });

    marcarBotaoAtivo(2);
    render();
  }

  function tryInitSim07Ep04(){
    var root = document.getElementById('sim-ep0704');
    if (root) initSim07Ep04(root); else setTimeout(tryInitSim07Ep04, 200);
  }
  tryInitSim07Ep04();
})();
</script>
""")

**Figura 7.4:** Simulatore EP07_04: Codice LBP di un Intorno 3×3


<figure id="fig-07-sim-ep0704">
  <img src="imagens/fig-07-sim-ep0704.png" alt=" Simulatore EP07_04: Codice LBP di un Intorno 3×3 " style="max-width:80%" />
  <figcaption><strong>Figura 7.4:</strong>  Simulatore EP07_04: Codice LBP di un Intorno 3×3 </figcaption>
</figure>

In [ ]:
%%writefile EP07_04.py
# Codice Python

In [ ]:
TestSuite("EP07_04.py").run()

### EP07_05 🔴 Istogramma delle Orientazioni di una Cella HOG

La funzione `hog` di `scikit-image`, impiegata nel progetto di classificazione delle cifre, suddivide l'immagine in piccole **celle** e, per ciascuna, costruisce un istogramma delle orientazioni del gradiente ponderato dalla magnitudine — esattamente la fase centrale descritta nella sezione sul descrittore HOG del capitolo.

Ti è stato affidato il compito di implementare questo calcolo per una singola cella, a partire dai valori di magnitudine e orientazione del gradiente **già calcolati** per ogni pixel della cella (omettendo il calcolo delle derivate parziali).

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $n$ (la cella ha $n \times n$ pixel) e $B$ (numero di contenitori dell'istogramma).
2. **Magnitudini:** Leggere $n$ righe con $n$ valori reali ciascuna, che rappresentano $|\nabla f(x,y)|$ per ogni pixel della cella.
3. **Orientazioni:** Leggere altre $n$ righe con $n$ valori reali ciascuna, che rappresentano $\theta(x,y)$ in **gradi**, già convertiti nell'intervallo **non orientato** $[0^\circ, 180^\circ)$, come convenzionalmente utilizzato dal HOG.
4. **Contenitori:** I $B$ contenitori coprono $[0^\circ, 180^\circ)$ in fasce uguali di larghezza $180/B$ gradi. Un pixel con orientazione $\theta$ appartiene al contenitore $\lfloor \theta / (180/B) \rfloor$; se questo indice è uguale a $B$ (possibile solo quando $\theta$ è esattamente $180^\circ$, il che non dovrebbe verificarsi secondo la direttiva del punto 3), utilizzare il contenitore $B-1$.
5. **Istogramma grezzo:** Per ogni pixel, accumulare la sua **magnitudine** (non il suo conteggio) nel contenitore corrispondente:
$$
H[b] = \sum_{(x,y)\, :\, \text{bin}(\theta(x,y)) = b} |\nabla f(x,y)|.
$$
6. **Normalizzazione L2:** Dopo aver costruito $H$, normalizzarlo per ottenere $\hat H$:
$$
\hat H[b] = \frac{H[b]}{\sqrt{\sum_{j=0}^{B-1} H[j]^2 + \epsilon}}, \qquad \epsilon = 10^{-6}.
$$
7. **Output:** Stampare l'istogramma grezzo $H$ (arrotondato a 2 cifre decimali) su una riga, seguito dall'istogramma normalizzato $\hat H$ (arrotondato a 4 cifre decimali) su un'altra riga, entrambi con i $B$ valori separati da spazi, nell'ordine dei contenitori.

#### 📌 Vincoli Computazionali

* ***Binning* non orientato:** l'intervallo delle orientazioni è $[0,180)$, non $[0,360)$ — i gradienti in direzioni opposte (differenza di $180^\circ$) contribuiscono allo **stesso** contenitore, convenzione standard del HOG per il rilevamento degli oggetti.
* **Accumulazione per magnitudine, non per conteggio:** l'istogramma pondera ogni pixel per la sua magnitudine di gradiente, non si limita a contare quanti pixel ricadono in ciascun contenitore.
* **Costante di stabilizzazione:** l'$\epsilon = 10^{-6}$ al denominatore della normalizzazione evita la divisione per zero quando la cella è completamente omogenea (tutte le magnitudini nulle).

#### 📐 Da dove provengono le matrici di input

Prima di questo EP, ogni pixel $(x,y)$ dell'immagine passa attraverso:

$$
G_x = f(x+1,y)-f(x-1,y), \qquad G_y = f(x,y+1)-f(x,y-1)
$$

$$
|\nabla f| = \sqrt{G_x^2+G_y^2}, \qquad \theta_{\text{sgn}} = \operatorname{atan2}(G_y,G_x)
$$

Poiché il HOG ignora la polarità del contrasto, l'angolo viene ripiegato nell'intervallo non orientato:

$$
\theta = \theta_{\text{sgn}} \bmod 180°
$$

Ripetendo questo per tutti i pixel di una cella $n\times n$, si ottengono le due matrici di input di questo esercizio: **magnitudini** $|\nabla f|$ e **orientazioni** $\theta \in [0°,180°)$.

#### 🧠 Fondamento Teorico

| Fase | Ruolo |
|---|---|
| Magnitudine del gradiente | Pondera il contributo di ogni pixel — i bordi forti pesano più del rumore debole |
| Orientazione non orientata | Rende il descrittore invariante alla polarità del contrasto (chiaro→scuro vs. scuro→chiaro) |
| Istogramma per cella | Riassume la distribuzione locale dei bordi in un vettore compatto |
| Normalizzazione L2 | Riduce la sensibilità del descrittore alle variazioni globali di illuminazione e contrasto |

La concatenazione degli istogrammi normalizzati di tutte le celle dell'immagine — non implementata in questo esercizio — forma il vettore di caratteristiche HOG completo, utilizzato come input del classificatore k-NN nel progetto del capitolo.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $n$ e $B$.
* Successive $n$ righe: $n$ magnitudini reali ciascuna.
* Successive $n$ righe: $n$ orientazioni reali (gradi, $[0,180)$) ciascuna.

**Output:**

* Riga 1: i $B$ valori dell'istogramma grezzo, arrotondati a 2 cifre decimali.
* Riga 2: i $B$ valori dell'istogramma normalizzato, arrotondati a 4 cifre decimali.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 2 2<br>1.0 2.0<br>3.0 4.0<br>10 100<br>170 20 | 5.00 5.00<br>0.7071 0.7071 | Bin di larghezza 90°: $[0,90)$ e $[90,180)$; le magnitudini 1 e 4 cadono nel bin 0, 2 e 3 nel bin 1. |
| 2 4<br>0.0 0.0<br>0.0 0.0<br>0 0<br>0 0 | 0.00 0.00 0.00 0.00<br>0.0000 0.0000 0.0000 0.0000 | Cella omogenea: l'$\epsilon$ evita la divisione per zero. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0705" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0705 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0705 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0705 button:hover { background: #e8dfcf; }
  #sim-ep0705 button.sim-ep0705_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0705_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0705_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_05: Istogramma delle orientazioni di una cella</span>
  <span class="sim-ep0705_pill">🔴 cella 3×3 fissa</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Regola B e osserva come la <b>matrice delle orientazioni</b> (indipendente da quella delle magnitudini) viene mappata
      nei contenitori tramite <code>bin = floor(θ / (180/B))</code>, e come le magnitudini vengono sommate in ciascun bin.
    </p>

    <!-- Controle B -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Numero di contenitori (B)</label>
        <span id="ep0705_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">2</span>
      </div>
      <input id="ep0705_sl" style="width:100%;accent-color:#2980b9;" max="6" min="2" step="1" type="range" value="2">
    </div>

    <!-- Entrada bruta (formato VPL) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📄 Input (esattamente come il programma legge da stdin)</div>
      <pre id="ep0705_stdin" style="background:#1e1e1e;color:#d4d4d4;border-radius:8px;padding:12px 14px;font-size:12px;line-height:1.5;overflow-x:auto;margin:0;"></pre>
    </div>

    <!-- Duas matrizes separadas -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🔢 Matrice delle magnitudini |∇f|</div>
        <div id="ep0705_mag_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📐 Matrice delle orientazioni θ (gradi) — colorata per bin</div>
        <div id="ep0705_ang_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
    </div>

    <!-- Regua 0-180 -->
    <div style="margin-bottom:22px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:10px;">📏 Dove ciascun θ cade sul righello [0°, 180°) — <code>bin = floor(θ / larghezza)</code></div>
      <div style="position:relative;height:70px;margin:0 6px;">
        <div id="ep0705_regua" style="position:absolute;top:28px;left:0;right:0;height:14px;border-radius:7px;overflow:hidden;display:flex;border:1px solid #d1d5db;"></div>
        <div id="ep0705_regua_ticks" style="position:absolute;top:44px;left:0;right:0;height:14px;"></div>
        <div id="ep0705_regua_marcas" style="position:absolute;top:0;left:0;right:0;height:26px;"></div>
      </div>
    </div>

    <!-- Faixas dos compartimentos -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📊 Intervalli di ciascun contenitore (larghezza = 180° / B)</div>
      <div id="ep0705_faixas" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
    </div>

    <!-- Grade de pixels colorida por bin (mag + ang juntos) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧩 Ogni pixel: magnitudine + orientazione → bin</div>
      <div id="ep0705_pixels" style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <!-- Botões -->
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:14px;">
      <button id="ep0705_btn_raw" class="ep0705_btn">Istogramma grezzo (H)</button>
      <button id="ep0705_btn_norm" class="ep0705_btn">Istogramma normalizzato (Ĥ)</button>
    </div>

    <!-- Barras -->
    <div id="ep0705_bars" style="display:flex;gap:6px;align-items:flex-end;height:120px;justify-content:center;margin-bottom:14px;"></div>

    <!-- Passo a passo -->
    <div style="margin-bottom:6px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧮 Calcolo passo passo (floor della divisione + somma delle magnitudini per bin)</div>
      <div id="ep0705_passos" style="background:#f3f4f6;border-radius:8px;padding:10px 12px;font-family:monospace;font-size:11px;color:#374151;line-height:1.8;"></div>
    </div>

    <div id="ep0705_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0705 .ep0705_btn { font-size:11px;padding:6px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0705 .ep0705_btn.ativo { background:#2980b9;color:#fff;border-color:#2980b9; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var n = 3;
    var mags = [[1.0,2.0,0.5],[3.0,4.0,1.5],[0.8,2.5,3.2]];
    var angs = [[10,100,45],[170,20,95],[60,150,5]];
    var CORES = ["#6366f1","#0ea5e9","#10b981","#f59e0b","#ef4444","#a855f7"];

    var slEl = root.querySelector("#ep0705_sl");
    var vlEl = root.querySelector("#ep0705_vl");
    var stdinEl = root.querySelector("#ep0705_stdin");
    var magGridEl = root.querySelector("#ep0705_mag_grid");
    var angGridEl = root.querySelector("#ep0705_ang_grid");
    var reguaEl = root.querySelector("#ep0705_regua");
    var reguaTicksEl = root.querySelector("#ep0705_regua_ticks");
    var reguaMarcasEl = root.querySelector("#ep0705_regua_marcas");
    var faixasEl = root.querySelector("#ep0705_faixas");
    var pxEl = root.querySelector("#ep0705_pixels");
    var bars = root.querySelector("#ep0705_bars");
    var passosEl = root.querySelector("#ep0705_passos");
    var dbg = root.querySelector("#ep0705_debug");
    var btnRaw = root.querySelector("#ep0705_btn_raw");
    var btnNorm = root.querySelector("#ep0705_btn_norm");
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle("ativo", !modoNorm);
      btnNorm.classList.toggle("ativo", modoNorm);

      var B = parseInt(slEl.value);
      vlEl.textContent = B;
      var largura = 180/B;

      // ---- Entrada bruta (stdin) ----
      var linhas = [];
      linhas.push(n + " " + B);
      mags.forEach(function(row){ linhas.push(row.map(function(v){return v.toFixed(1);}).join(" ")); });
      angs.forEach(function(row){ linhas.push(row.join(" ")); });
      stdinEl.textContent = linhas.join("\\n");

      // ---- bin de cada pixel (floor(theta/largura), clip) ----
      var binsMat = [];
      for(var i=0;i<n;i++){
        binsMat.push([]);
        for(var j=0;j<n;j++){
          var raw = angs[i][j]/largura;
          var b = Math.floor(raw);
          if(b > B-1) b = B-1;
          if(b < 0) b = 0;
          binsMat[i].push(b);
        }
      }

      // ---- Matriz de magnitudes (grid simples) ----
      magGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var d = document.createElement("div");
          d.style.cssText = "text-align:center;border-radius:8px;padding:8px 4px;font-size:12px;font-family:monospace;background:#f9fafb;border:1px solid #e5e7eb;color:#374151;";
          d.textContent = mags[i][j].toFixed(1);
          magGridEl.appendChild(d);
        }
      }

      // ---- Matriz de orientações (colorida por bin, com floor explícito) ----
      angGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var b2 = binsMat[i][j];
          var cor2 = CORES[b2];
          var raw2 = angs[i][j]/largura;
          var d2 = document.createElement("div");
          d2.style.cssText = "text-align:center;border-radius:8px;padding:6px 4px;font-size:11px;font-family:monospace;background:"+cor2+"22;border:2px solid "+cor2+";color:#374151;";
          d2.innerHTML = "<div style=\\"font-weight:700;\\">"+angs[i][j]+"°</div>"+
            "<div style=\\"font-size:9px;color:#6b7280;\\">÷"+largura.toFixed(1)+"="+raw2.toFixed(2)+"</div>"+
            "<div style=\\"font-size:9px;font-weight:700;color:"+cor2+";\\">⌊·⌋=bin "+b2+"</div>";
          angGridEl.appendChild(d2);
        }
      }

      // ---- Régua 0-180 com faixas coloridas ----
      reguaEl.innerHTML = "";
      for(var b3=0;b3<B;b3++){
        var seg = document.createElement("div");
        seg.style.cssText = "flex:1;background:"+CORES[b3]+";opacity:0.35;border-right:1px solid rgba(255,255,255,0.6);";
        reguaEl.appendChild(seg);
      }
      // ticks (limites dos bins)
      reguaTicksEl.innerHTML = "";
      for(var b4=0;b4<=B;b4++){
        var pct = (b4*largura/180*100);
        var tick = document.createElement("div");
        tick.style.cssText = "position:absolute;left:"+pct+"%;top:0;font-size:9px;color:#6b7280;transform:translateX(-50%);white-space:nowrap;";
        tick.textContent = (b4*largura).toFixed(0)+"°";
        reguaTicksEl.appendChild(tick);
      }
      // marcadores dos angulos de cada pixel
      reguaMarcasEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var ang = angs[i][j];
          var b5 = binsMat[i][j];
          var pctm = (ang/180*100);
          var marker = document.createElement("div");
          marker.style.cssText = "position:absolute;left:"+pctm+"%;top:0;transform:translateX(-50%);display:flex;flex-direction:column;align-items:center;";
          marker.innerHTML = "<div style=\\"font-size:9px;color:"+CORES[b5]+";font-weight:700;\\">("+i+","+j+")</div>"+
            "<div style=\\"width:0;height:0;border-left:5px solid transparent;border-right:5px solid transparent;border-top:8px solid "+CORES[b5]+";\\"></div>";
          reguaMarcasEl.appendChild(marker);
        }
      }

      // ---- Faixas dos bins (legenda) ----
      faixasEl.innerHTML = "";
      for(var b=0;b<B;b++){
        var lo = (b*largura).toFixed(1);
        var hi = ((b+1)*largura).toFixed(1);
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:6px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:20px;padding:4px 10px;font-size:11px;color:#374151;";
        chip.innerHTML = "<span style=\\"width:10px;height:10px;border-radius:50%;background:"+CORES[b]+";display:inline-block;\\"></span>bin "+b+": ["+lo+"°, "+hi+"°)";
        faixasEl.appendChild(chip);
      }

      // ---- Atribuição por pixel + histograma bruto ----
      var H = new Array(B).fill(0);
      var binsPorPixel = [];
      pxEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var mag = mags[i][j], ang = angs[i][j];
          var bin = binsMat[i][j];
          binsPorPixel.push({i:i, j:j, mag:mag, ang:ang, bin:bin});
          H[bin] += mag;

          var div = document.createElement("div");
          var cor = CORES[bin];
          div.style.cssText = "text-align:center;border-radius:10px;padding:8px 6px;font-size:11px;background:"+cor+"22;border:2px solid "+cor+";color:#374151;";
          div.innerHTML = "<div style=\\"font-weight:700;\\">mag="+mag.toFixed(1)+"</div>"+
            "<div style=\\"font-family:monospace;\\">θ="+ang+"°</div>"+
            "<div style=\\"font-weight:700;color:"+cor+";\\">→ bin "+bin+"</div>";
          pxEl.appendChild(div);
        }
      }

      var denom = Math.sqrt(H.reduce(function(s,v){return s+v*v;},0) + 1e-6);
      var Hn = H.map(function(v){ return v/denom; });

      // ---- Barras (coloridas por bin) ----
      var dados = modoNorm ? Hn : H;
      var maxD = Math.max.apply(null, dados.concat([0.001]));
      bars.innerHTML = "";
      dados.forEach(function(v, b){
        var col = document.createElement("div");
        col.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:4px;";
        var barra = document.createElement("div");
        var altura = Math.round((v/maxD)*90) + 4;
        barra.style.cssText = "width:34px;height:"+altura+"px;background:"+CORES[b]+";border-radius:4px 4px 0 0;";
        var label = document.createElement("div");
        label.style.cssText = "font-family:monospace;font-size:10px;color:#4b5563;";
        label.textContent = modoNorm ? v.toFixed(4) : v.toFixed(2);
        var binLabel = document.createElement("div");
        binLabel.style.cssText = "font-size:9px;color:#9ca3af;";
        binLabel.textContent = "bin "+b;
        col.appendChild(barra);
        col.appendChild(label);
        col.appendChild(binLabel);
        bars.appendChild(col);
      });

      // ---- Passo a passo (floor + soma) ----
      var passos = [];
      for(var b=0;b<B;b++){
        var contribs = binsPorPixel.filter(function(p){ return p.bin===b; });
        var termos = contribs.map(function(p){ return p.mag.toFixed(2)+" (θ="+p.ang+"°→⌊"+(p.ang/largura).toFixed(2)+"⌋="+p.bin+")"; }).join(" + ");
        if(termos === "") termos = "(nenhum pixel)";
        passos.push("<span style=\\"color:"+CORES[b]+";font-weight:700;\\">H["+b+"]</span> = "+termos+" = <b>"+H[b].toFixed(2)+"</b>");
      }
      passosEl.innerHTML = passos.join("<br>");

      dbg.textContent = "H=[" + H.map(function(v){return v.toFixed(2);}).join(", ") + "]  |  Ĥ=[" +
        Hn.map(function(v){return v.toFixed(4);}).join(", ") + "]";
    }

    slEl.addEventListener("input", render);
    btnRaw.addEventListener("click", function(){ modoNorm = false; render(); });
    btnNorm.addEventListener("click", function(){ modoNorm = true; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0705");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.5:** Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli in bin)


<figure id="fig-07-sim-ep0705">
  <img src="imagens/fig-07-sim-ep0705.png" alt=" Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli in bin) " style="max-width:80%" />
  <figcaption><strong>Figura 7.5:</strong>  Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli in bin) </figcaption>
</figure>

In [ ]:
%%writefile EP07_05.py
# Codice Python

In [ ]:
TestSuite("EP07_05.py").run()

### EP07_06 🟣 *Pipeline* Completo: Descrittori + k-NN + Valutazione Multi-Classe

Questo esercizio integra le tre fasi centrali del capitolo in un unico *pipeline*, riproducendo in miniatura il **Progetto Pratico 2** (classificazione di texture sintetiche tramite LBP): un insieme di istogrammi di descrittori **già estratti** (come se fossero istogrammi LBP) viene utilizzato per addestrare un classificatore k-NN, che a sua volta viene valutato su un insieme di test indipendente mediante una matrice di confusione multi-classe.

A differenza dell'EP07_01, qui lo spazio delle caratteristiche ha dimensione arbitraria $H$ (la dimensione dell'istogramma), esistono più di due classi e la metrica di distanza è un parametro di input — consentendo di riprodurre l'esperimento di confronto delle metriche discusso nel capitolo.

#### 📋 Linee Guida di Implementazione

1. **Classi:** Leggere l'intero $C$ (numero di classi) seguito da $C$ nomi di classe (*stringhe* senza spazi), nell'ordine in cui devono apparire nella matrice di confusione.
2. **Configurazione:** Leggere l'intero $H$ (dimensione degli istogrammi), la *stringa* $M$ (metrica: `euclidiana` o `manhattan`) e l'intero dispari $k$.
3. **Addestramento:** Leggere l'intero $N$ e, successivamente, $N$ righe, ciascuna contenente il nome della classe seguito da $H$ valori reali (l'istogramma del descrittore).
4. **Test:** Leggere l'intero $Q$ e, successivamente, $Q$ righe, ciascuna contenente il nome della classe **reale** seguito da $H$ valori reali (l'istogramma del descrittore del campione di test).
5. **Distanza:** Per ogni campione di test, calcolare la distanza da ogni esempio di addestramento utilizzando la metrica $M$:
$$
d_{\text{euclidiana}}(u,v) = \sqrt{\sum_{j=1}^{H}(u_j-v_j)^2}, \qquad
d_{\text{manhattan}}(u,v) = \sum_{j=1}^{H} |u_j - v_j|.
$$
6. **Classificazione k-NN:** Selezionare i $k$ esempi di addestramento più vicini (pareggio di distanza risolto dall'ordine di lettura, come nell'EP07_01) e classificare mediante la classe maggioritaria tra essi. In caso di **pareggio di voti** tra due o più classi, scegliere quella che appare **per prima** nella lista delle classi del punto 1.
7. **Matrice di confusione:** Costruire una matrice $C \times C$ in cui la riga corrisponde alla classe reale e la colonna alla classe prevista, seguendo l'ordine delle classi del punto 1.
8. **Accuratezza:** Calcolare l'accuratezza globale come rapporto tra successi e $Q$.
9. **Output:** Per ogni campione di test, nell'ordine di input, stampare la classe prevista. Successivamente, stampare la matrice di confusione (una riga per classe reale, valori separati da spazi, nell'ordine delle classi). Infine, stampare l'accuratezza arrotondata a 4 cifre decimali.

#### 📌 Vincoli Computazionali

* **Metrica selezionabile:** implementare entrambe le distanze; la metrica $M$ definisce quale viene utilizzata in tutta l'esecuzione (non è possibile mescolare metriche nella stessa chiamata).
* **Pareggio di voti deterministico:** il criterio del punto 6 (ordine della lista delle classi) deve essere seguito anche quando il pareggio coinvolge più di due classi.
* **Indipendenza tra addestramento e test:** non è necessario verificare che i campioni di test non appaiano nell'addestramento — si assume che l'input sia valido.

#### 🧠 Fondamento Teorico

| Fase dell'esercizio | Fase corrispondente nel capitolo |
|---|---|
| Istogrammi di addestramento/test già estratti | `descritor_lbp` applicato alle texture sintetiche |
| Distanza euclidea o Manhattan | Parametro `metric` del `KNeighborsClassifier` |
| Votazione maggioritaria con $k$ vicini | `KNeighborsClassifier.predict` |
| Matrice di confusione $C\times C$ | `confusion_matrix` di `scikit-learn` |
| Accuratezza globale | `accuracy_score` di `scikit-learn` |

Questo esercizio evidenzia, in modo controllato, un risultato discusso nel capitolo: la **scelta della metrica di distanza** e del **valore di $k$** può cambiare la classe prevista per lo stesso campione, anche mantenendo fisso il descrittore utilizzato — rafforzando l'idea che, nel riconoscimento di pattern classico, il descrittore, la metrica e il classificatore formano un sistema interdipendente, e non componenti isolate.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $C$ seguito da $C$ nomi di classe.
* Riga 2: intero $H$, *stringa* $M$ e intero $k$.
* Riga 3: intero $N$.
* Prossime $N$ righe di addestramento: nome della classe seguito da $H$ reali.
* Riga successiva: intero $Q$.
* Prossime $Q$ righe di test: nome della classe reale seguito da $H$ reali.

**Output:**

* $Q$ righe con la classe prevista di ogni campione di test, nell'ordine di input.
* $C$ righe con la matrice di confusione (una riga per classe reale).
* Ultima riga: `Acuracia: <valore>`.

#### 📌 Esempi

| Input (riassunto) | Output | Osservazione |
|---|---|---|
| 2 granular listrada<br>2 euclidiana 1<br>4<br>granular 0.9 0.1<br>granular 0.8 0.2<br>listrada 0.1 0.9<br>listrada 0.2 0.8<br>2<br>granular 0.85 0.15<br>listrada 0.15 0.85 | granular<br>listrada<br>1 0<br>0 1<br>Acuracia: 1.0000 | Con $k=1$, ogni test viene classificato dal vicino di addestramento più prossimo. |

> ### 📝 Nota
>
> Questo simulatore utilizza un insieme semplificato di **3 classi** (`granulare`, `striata`, `maculata`) su punti 2D fittizi, solo per illustrare il *pipeline* di votazione, spareggio e matrice di confusione del k-NN. Nell'**EP07_07**, applicherai questa stessa logica a un mosaico di un'immagine reale, che introduce una quarta classe (`scacchiera`) e sostituisce i punti 2D con istogrammi LBP estratti direttamente dai pixel dell'immagine.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0706" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0706 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0706 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0706 button:hover { background: #e8dfcf; }
  #sim-ep0706 button.sim-ep0706_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0706_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0706_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_06: Pipeline k-NN Multi-Classe</span>
  <span class="sim-ep0706_pill">6 Training &middot; 3 Test &middot; 3 Classi</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Scegli la metrica, il valore di k e il campione di test (★). Vedi i k vicini più prossimi, la votazione,
      il pareggio quando necessario, e come questo si propaga alla matrice di confusione e all'accuratezza dell'intero insieme.
    </p>

    <!-- Controles -->
    <div style="display:flex;flex-wrap:wrap;gap:18px;justify-content:center;margin-bottom:16px;">
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Metrica (M)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_be" class="ep0706_btn">Euclidea</button>
          <button id="ep0706_bm" class="ep0706_btn">Manhattan</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Vicini (k)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_k1" class="ep0706_btn">k=1</button>
          <button id="ep0706_k3" class="ep0706_btn">k=3</button>
          <button id="ep0706_k5" class="ep0706_btn">k=5</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Campione di test (★)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_t0" class="ep0706_btn">test 1</button>
          <button id="ep0706_t1" class="ep0706_btn">test 2</button>
          <button id="ep0706_t2" class="ep0706_btn">test 3</button>
        </div>
      </div>
    </div>

    <!-- Legenda -->
    <div id="ep0706_legenda" style="display:flex;gap:10px;justify-content:center;margin-bottom:10px;"></div>

    <!-- Dispersao 2D -->
    <div style="max-width:340px;margin:0 auto 16px auto;height:300px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa;">
      <div id="ep0706_svg_container" style="width:100%;height:100%;"></div>
    </div>

    <!-- Distancias ordenadas -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📏 Distanze dal campione di test (ordinate) — <span style="font-weight:400;font-size:10px;color:#8a8672;">#i = ordine di lettura nella lista di training (passa il mouse)</span></div>
      <div id="ep0706_dists" style="display:grid;grid-template-columns:1fr 1fr;gap:2px 10px;font-family:monospace;font-size:10px;"></div>
    </div>

    <!-- Votacao -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🗳️ Votazione tra i k vicini</div>
      <div id="ep0706_votos" style="display:flex;gap:10px;justify-content:center;margin-bottom:6px;"></div>
      <div id="ep0706_previsao" style="text-align:center;font-size:12px;font-weight:bold;"></div>
    </div>

    <!-- Matriz de confusao + acuracia (conjunto de teste inteiro) -->
    <div style="margin-bottom:8px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📋 Matrice di confusione e accuratezza — eseguendo il pipeline sui 3 campioni di test</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;justify-content:center;">
        <table id="ep0706_cm" style="border-collapse:collapse;font-size:11px;font-family:monospace;"></table>
        <div id="ep0706_acc" style="font-size:13px;font-weight:bold;color:#5e5a4a;"></div>
      </div>
    </div>

    <div id="ep0706_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0706 .ep0706_btn { font-size:11px;padding:5px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0706 .ep0706_btn.ativo { background:#7c3aed;color:#fff;border-color:#7c3aed; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var classes = ["granular","listrada","manchada"];
    var CORES = {granular:"#6366f1", listrada:"#f59e0b", manchada:"#10b981"};

    var trainPts = [
      {nome:"granular_1", cls:"granular", x:0.70, y:0.70},
      {nome:"granular_2", cls:"granular", x:0.25, y:0.85},
      {nome:"listrada_1", cls:"listrada", x:0.85, y:0.50},
      {nome:"listrada_2", cls:"listrada", x:0.60, y:0.15},
      {nome:"manchada_1", cls:"manchada", x:0.30, y:0.30},
      {nome:"manchada_2", cls:"manchada", x:0.15, y:0.55}
    ];
    var testPts = [
      {nome:"teste 1", cls:"granular", x:0.50, y:0.50},
      {nome:"teste 2", cls:"listrada", x:0.70, y:0.20},
      {nome:"teste 3", cls:"manchada", x:0.20, y:0.40}
    ];

    var svgContainer = root.querySelector("#ep0706_svg_container");
    var svg = svgNS("svg");
    svg.setAttribute("viewBox", "0 0 100 100");
    svg.setAttribute("style", "width:100%;height:100%;");
    svgContainer.appendChild(svg);
    var legendaEl = root.querySelector("#ep0706_legenda");
    var distsEl = root.querySelector("#ep0706_dists");
    var votosEl = root.querySelector("#ep0706_votos");
    var previsaoEl = root.querySelector("#ep0706_previsao");
    var cmEl = root.querySelector("#ep0706_cm");
    var accEl = root.querySelector("#ep0706_acc");
    var dbg = root.querySelector("#ep0706_debug");

    var be = root.querySelector("#ep0706_be"), bm = root.querySelector("#ep0706_bm");
    var bk1 = root.querySelector("#ep0706_k1"), bk3 = root.querySelector("#ep0706_k3"), bk5 = root.querySelector("#ep0706_k5");
    var bt0 = root.querySelector("#ep0706_t0"), bt1 = root.querySelector("#ep0706_t1"), bt2 = root.querySelector("#ep0706_t2");

    var metrica = "euclidiana", k = 1, testSel = 0;

    function dist(u, v){
      var dx = u.x-v.x, dy = u.y-v.y;
      if(metrica === "euclidiana") return Math.sqrt(dx*dx+dy*dy);
      return Math.abs(dx)+Math.abs(dy);
    }

    function knnPredict(xtest){
      var ds = trainPts.map(function(p, i){ return {i:i, p:p, d:dist(xtest, p)}; });
      ds.sort(function(a,b){ return a.d - b.d; }); // ordem estavel = desempate por ordem de leitura
      var viz = ds.slice(0, k);
      var votos = {}; classes.forEach(function(c){ votos[c]=0; });
      viz.forEach(function(v){ votos[v.p.cls]++; });
      var maxV = Math.max.apply(null, classes.map(function(c){return votos[c];}));
      var empatados = classes.filter(function(c){ return votos[c]===maxV; });
      var pred = empatados[0]; // primeira classe da lista entre as empatadas
      return {pred:pred, viz:viz, votos:votos, empatados:empatados, ordenados:ds};
    }

    function svgNS(tag){
      // Concatenado de propósito: evita que filtros de auto-link do Moodle
      // reconheçam "http://www.w3.org/2000/svg" como URL e insiram uma tag <a>
      // dentro desta string, o que quebraria a sintaxe do createElementNS.
      var SVG_NS = "http" + "://www.w3.org/2000/svg";
      return document.createElementNS(SVG_NS, tag);
    }

    function render(){
      be.classList.toggle("ativo", metrica==="euclidiana");
      bm.classList.toggle("ativo", metrica==="manhattan");
      bk1.classList.toggle("ativo", k===1);
      bk3.classList.toggle("ativo", k===3);
      bk5.classList.toggle("ativo", k===5);
      bt0.classList.toggle("ativo", testSel===0);
      bt1.classList.toggle("ativo", testSel===1);
      bt2.classList.toggle("ativo", testSel===2);

      // Legenda
      legendaEl.innerHTML = "";
      classes.forEach(function(c){
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;";
        chip.innerHTML = '<span style="width:10px;height:10px;border-radius:50%;background:'+CORES[c]+';display:inline-block;"></span>'+c;
        legendaEl.appendChild(chip);
      });

      var xt = testPts[testSel];
      var r = knnPredict(xt);
      var vizIdx = r.viz.map(function(v){ return v.i; });

      // ---- SVG: pontos de treino, linhas para vizinhos, estrela de teste ----
      svg.innerHTML = "";
      // grade leve
      for(var g=1; g<4; g++){
        var lineV = svgNS("line");
        lineV.setAttribute("x1", g*25); lineV.setAttribute("y1", 0);
        lineV.setAttribute("x2", g*25); lineV.setAttribute("y2", 100);
        lineV.setAttribute("stroke", "#eee"); lineV.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineV);
        var lineH = svgNS("line");
        lineH.setAttribute("x1", 0); lineH.setAttribute("y1", g*25);
        lineH.setAttribute("x2", 100); lineH.setAttribute("y2", g*25);
        lineH.setAttribute("stroke", "#eee"); lineH.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineH);
      }
      // linhas ate os vizinhos (desenhadas antes dos pontos, para ficarem por baixo)
      vizIdx.forEach(function(i){
        var p = trainPts[i];
        var line = svgNS("line");
        line.setAttribute("x1", xt.x*100); line.setAttribute("y1", (1-xt.y)*100);
        line.setAttribute("x2", p.x*100); line.setAttribute("y2", (1-p.y)*100);
        line.setAttribute("stroke", CORES[p.cls]); line.setAttribute("stroke-width", "0.6");
        line.setAttribute("stroke-dasharray", "1.5,1"); line.setAttribute("opacity", "0.7");
        svg.appendChild(line);
      });
      // pontos de treino
      trainPts.forEach(function(p, i){
        var isViz = vizIdx.indexOf(i) !== -1;
        if(isViz){
          var halo = svgNS("circle");
          halo.setAttribute("cx", p.x*100); halo.setAttribute("cy", (1-p.y)*100);
          halo.setAttribute("r", 5); halo.setAttribute("fill", "none");
          halo.setAttribute("stroke", CORES[p.cls]); halo.setAttribute("stroke-width", "0.8");
          svg.appendChild(halo);
        }
        var c = svgNS("circle");
        c.setAttribute("cx", p.x*100); c.setAttribute("cy", (1-p.y)*100);
        c.setAttribute("r", 3.2);
        c.setAttribute("fill", CORES[p.cls]);
        c.setAttribute("stroke", "#fff"); c.setAttribute("stroke-width", "0.6");
        c.setAttribute("opacity", isViz ? "1" : "0.55");
        svg.appendChild(c);
      });
      // estrela de teste
      var correto = (r.pred === xt.cls);
      var estCor = correto ? "#16a34a" : "#dc2626";
      var halo2 = svgNS("circle");
      halo2.setAttribute("cx", xt.x*100); halo2.setAttribute("cy", (1-xt.y)*100);
      halo2.setAttribute("r", 5.5); halo2.setAttribute("fill", "#fff");
      halo2.setAttribute("stroke", estCor); halo2.setAttribute("stroke-width", "0.8");
      svg.appendChild(halo2);
      var txt = svgNS("text");
      txt.setAttribute("x", xt.x*100); txt.setAttribute("y", (1-xt.y)*100+1.8);
      txt.setAttribute("text-anchor", "middle"); txt.setAttribute("font-size", "6.5");
      txt.setAttribute("fill", estCor);
      txt.textContent = "★";
      svg.appendChild(txt);

      // ---- Distancias ordenadas ----
      distsEl.innerHTML = "";
r.ordenados.forEach(function(v, ord){
  var dentroK = ord < k;
  var row = document.createElement("div");
  row.style.cssText = "display:flex;justify-content:space-between;align-items:center;padding:2px 6px;border-radius:6px;" +
    (dentroK ? "background:"+CORES[v.p.cls]+"22;border:1px solid "+CORES[v.p.cls]+";" : "background:#f9fafb;border:1px solid #f1f1f1;color:#9ca3af;");
  row.innerHTML =
    '<span style="display:flex;align-items:center;gap:4px;">' +
      (dentroK ? '✓' : '\u00A0') +
      '<span title="posizione di lettura nella lista originale di training — usata per il pareggio quando due distanze sono uguali" ' +
        'style="background:#eee;color:#9ca3af;border-radius:3px;padding:0 3px;font-size:8.5px;cursor:help;">#' + (v.i+1) + '</span>' +
      ' ' + v.p.nome + ' <span style="color:'+CORES[v.p.cls]+';font-weight:700;">('+v.p.cls+')</span>' +
    '</span>' +
    '<span>d='+v.d.toFixed(4)+'</span>';
  distsEl.appendChild(row);
});

      // ---- Votacao ----
      votosEl.innerHTML = "";
      classes.forEach(function(c){
        var venceu = (c === r.pred);
        var empatou = r.empatados.length > 1 && r.empatados.indexOf(c) !== -1;
        var div = document.createElement("div");
        div.style.cssText = "text-align:center;border-radius:10px;padding:8px 14px;font-size:12px;" +
          (venceu ? "background:"+CORES[c]+"22;border:2px solid "+CORES[c]+";" : "background:#f9fafb;border:1px solid #e5e7eb;color:#9ca3af;");
        div.innerHTML = '<div style="font-weight:700;color:'+CORES[c]+';">'+c+'</div><div style="font-size:16px;font-weight:700;">'+r.votos[c]+'</div>' +
          (empatou ? '<div style="font-size:9px;color:#b91c1c;">empate</div>' : '');
        votosEl.appendChild(div);
      });
      var msgEmpate = r.empatados.length > 1 ? " (empate entre "+r.empatados.join(", ")+" — desempate pela ordem da lista de classes)" : "";
      previsaoEl.innerHTML = 'Classe prevista: <span style="color:'+CORES[r.pred]+';">'+r.pred+'</span>' + msgEmpate +
        ' &nbsp;|&nbsp; classe real: <span style="color:'+CORES[xt.cls]+';">'+xt.cls+'</span> ' + (correto ? '✅' : '❌');

      // ---- Matriz de confusao + acuracia sobre as 3 amostras de teste ----
      var cm = [[0,0,0],[0,0,0],[0,0,0]];
      var acertos = 0;
      var predsGlobais = [];
      testPts.forEach(function(tp){
        var rr = knnPredict(tp);
        predsGlobais.push(rr.pred);
        var iReal = classes.indexOf(tp.cls);
        var iPrev = classes.indexOf(rr.pred);
        cm[iReal][iPrev]++;
        if(rr.pred === tp.cls) acertos++;
      });
      var acc = acertos/testPts.length;

      var thead = '<tr><td></td>' + classes.map(function(c){ return '<td style="padding:4px 8px;color:'+CORES[c]+';font-weight:700;">'+c.slice(0,4)+'</td>'; }).join('') + '</tr>';
      var rows = classes.map(function(cReal, i){
        var cells = classes.map(function(cPrev, j){
          var v = cm[i][j];
          var diag = (i===j);
          var bg = v===0 ? '#fff' : (diag ? '#dcfce7' : '#fee2e2');
          return '<td style="padding:4px 10px;text-align:center;border:1px solid #e5e7eb;background:'+bg+';">'+v+'</td>';
        }).join('');
        return '<tr><td style="padding:4px 8px;color:'+CORES[cReal]+';font-weight:700;">'+cReal.slice(0,4)+'</td>'+cells+'</tr>';
      }).join('');
      cmEl.innerHTML = thead + rows;
      accEl.textContent = "Accuratezza: " + acc.toFixed(4) + " (" + acertos + "/" + testPts.length + ")";

      dbg.textContent = "M="+metrica+" k="+k+" | teste_sel="+xt.nome+" | y_pred(todas)=["+predsGlobais.join(", ")+"]";
    }

    be.addEventListener("click", function(){ metrica="euclidiana"; render(); });
    bm.addEventListener("click", function(){ metrica="manhattan"; render(); });
    bk1.addEventListener("click", function(){ k=1; render(); });
    bk3.addEventListener("click", function(){ k=3; render(); });
    bk5.addEventListener("click", function(){ k=5; render(); });
    bt0.addEventListener("click", function(){ testSel=0; render(); });
    bt1.addEventListener("click", function(){ testSel=1; render(); });
    bt2.addEventListener("click", function(){ testSel=2; render(); });

    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0706");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.6:** Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione)


<figure id="fig-07-sim-ep0706">
  <img src="imagens/fig-07-sim-ep0706.png" alt=" Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione) " style="max-width:80%" />
  <figcaption><strong>Figura 7.6:</strong>  Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione) </figcaption>
</figure>

In [ ]:
%%writefile EP07_06.py
# Codice Python

In [ ]:
TestSuite("EP07_06.py").run()

### EP07_07 ⚫ Classificazione Reale di un Mosaico di Trame tramite LBP + k-NN

Negli esercizi precedenti, il descrittore LBP (**EP07_04**) e il classificatore k-NN multiclasse (**EP07_06**) sono stati studiati separatamente, sempre a partire da dati già forniti in input — intorni $3\times3$ isolati o istogrammi precedentemente estratti. In questo esercizio conclusivo del capitolo, il programma dovrà **leggere un'immagine reale**, nel formato **PGM ASCII (P2)**, calcolare il descrittore LBP direttamente dai pixel e, successivamente, classificare ciascuna regione tramite il k-NN, riproducendo, in scala ridotta, il flusso completo di un sistema di riconoscimento delle trame. Questo approccio anticipa anche l'idea di **classificazione per mosaico di regioni**, legata alla segmentazione semantica studiata in un capitolo successivo.

Il simulatore interattivo dell'**EP07_06** utilizzava solo tre classi (`granulare`, `a strisce` e `chiazzata`) rappresentate da punti bidimensionali fittizi. In questo esercizio, si aggiunge una quarta classe, **a scacchi**, e i punti vengono sostituiti da istogrammi LBP estratti da un'immagine reale.

L'immagine di input è un **mosaico** formato da una griglia $G\times G$ di blocchi quadrati di $S\times S$ pixel. Ogni blocco contiene un campione di una delle quattro classi di trama sintetica del capitolo: **granulare**, **a strisce**, **chiazzata** o **a scacchi** (motivo a scacchiera con intensità alternate). Come negli altri esercizi del libro, il caricamento dell'immagine è effettuato dalla funzione didattica `mm.readImg`.

> ### 💡 Perché un singolo mosaico, e non più immagini?
>
> L'input riunisce i $G \times G$ campioni di trama in un unico file **PGM**, solo per semplificare la lettura dei dati ed evitare l'apertura di più file. Per l'algoritmo, ciò non modifica l'elaborazione: ogni blocco viene trattato in modo indipendente, come se fosse un'immagine isolata.
> L'unica eccezione è l'**esclusione del bordo** (punto 4 di seguito).

#### 📋 Linee Guida di Implementazione

1. **Lettura delle dimensioni dell'immagine**

   Leggere, tramite l'input standard, due righe contenenti rispettivamente il numero di righe $L$ e il numero di colonne $C$ del mosaico (entrambi multipli della dimensione del blocco $S$, con $L=C$).

2. **Caricamento dell'immagine**

   Utilizzare la funzione didattica

   ```python
   f = mm.readImg(L, C)
   ```

   per leggere i valori di intensità $L \times C$ (toni di grigio, `uint8`) del mosaico.

3. **Parametri della griglia**

   Leggere l'intero $G$ (numero di blocchi per lato) e l'intero $S$ (dimensione del lato di ciascun blocco, in pixel), soddisfacendo $L = C = G \times S$.

4. **Calcolo del codice LBP per pixel**

   Per ogni pixel **interno** dell'immagine (cioè che non si trova sul bordo globale di `f` — riga o colonna $0$ o $L-1$/$C-1$), calcolare il codice LBP con $P=8$ vicini e raggio $R=1$, percorrendo i vicini in senso **orario** a partire dall'angolo superiore sinistro, esattamente come nell'EP07_04: `[riga-1][colonna-1]`, `[riga-1][colonna]`, `[riga-1][colonna+1]`, `[riga][colonna+1]`, `[riga+1][colonna+1]`, `[riga+1][colonna]`, `[riga+1][colonna-1]`, `[riga][colonna-1]`.

   I pixel sul bordo globale dell'immagine **non** possiedono un intorno completo e devono essere **ignorati** (non contribuiscono a nessun istogramma). Ciò include i pixel di bordo che ricadono all'interno di un blocco (l'esclusione è sempre relativa al bordo dell'intera immagine, non al bordo di ciascun blocco).

5. **Istogramma LBP uniforme per blocco (10 contenitori)**

   Per ogni blocco $(i,j)$ della griglia ($i,j = 0,\ldots,G-1$), accumulare, tra i suoi pixel validi (punto 4), un istogramma $H^{(i,j)}$ di $10$ contenitori:

   * Considerando la sequenza circolare di bit $s_0,\ldots,s_7$ del pixel (stessa regola di transizioni dell'EP07_04): se il numero di transizioni è $\le 2$ (pattern **uniforme**), il pixel contribuisce al contenitore $\operatorname{popcount}(s_0,\ldots,s_7) \in \{0,\ldots,8\}$ (numero di bit uguali a `1`);
   * Altrimenti (pattern **non uniforme**), il pixel contribuisce al contenitore $9$.

   Alla fine, normalizzare l'istogramma di ciascun blocco dividendolo per il numero di pixel validi in esso contenuti, ottenendo $\hat H^{(i,j)}$, con $\sum_{b=0}^{9} \hat H^{(i,j)}[b] = 1$.

6. **Prototipi di addestramento**

   Leggere l'intero $Ncl$ (numero di classi) seguito da $Ncl$ nomi di classe (ordine che definisce la matrice di confusione e il pareggio di votazione, come nell'EP07_06); successivamente, leggere la *stringa* $M$ (metrica: `euclidiana` o `manhattan`) e l'intero dispari $k$; infine, leggere l'intero $N$ (numero di prototipi) e, per ciascuno, il nome della classe seguito da $10$ valori reali (istogramma prototipo già normalizzato).

7. **Classificazione k-NN di ogni blocco**

   Per ogni blocco, calcolare la distanza di $\hat H^{(i,j)}$ da ciascuno degli $N$ prototipi, utilizzando la metrica $M$ (stesse formule dell'EP07_06). Selezionare i $k$ prototipi più vicini (pareggio di distanza risolto dall'ordine di lettura dei prototipi) e classificare tramite la classe maggioritaria (pareggio di votazione risolto dall'ordine delle classi del punto 6).

8. **Etichette reali e valutazione**

   Leggere, in un'unica riga, i $G \times G$ nomi di classe **reali** di ciascun blocco, in ordine di lettura per riga della griglia (blocco $(0,0)$, $(0,1)$, …, $(0,G-1)$, $(1,0)$, …). Costruire la matrice di confusione $Ncl \times Ncl$ (riga = classe reale, colonna = classe prevista) e l'accuratezza globale.

9. **Output**

   Stampare, per ogni blocco (nello stesso ordine di lettura delle etichette reali del punto 8), la classe prevista. Successivamente, stampare la matrice di confusione (una riga per classe reale, nell'ordine del punto 6). Infine, stampare l'accuratezza, arrotondata a 4 cifre decimali.

#### 📌 Vincoli Computazionali

* **Descrittore fisso:** $P=8$, $R=1$ e $10$ contenitori (come da punto 5) sono fissi in questo esercizio — non vengono letti dall'input.
* **Esclusione del bordo globale, non del blocco:** un pixel sul confine tra due blocchi, ma interno all'immagine, è valido e contribuisce normalmente all'istogramma del blocco a cui appartiene.
* **Ordine di lettura come criterio di pareggio:** sia il pareggio di distanza (punto 7) sia quello di votazione (punto 7) seguono esattamente le stesse convenzioni dell'EP07_01 e dell'EP07_06.
* **Prototipi come input, non appresi:** diversamente dal Progetto Pratico 2, gli istogrammi di addestramento sono forniti direttamente in input; il programma non deve generare trame sintetiche.

#### 🧠 Fondamenti Teorici

| Fase dell'esercizio | Fase corrispondente nel capitolo |
|---|---|
| Lettura dell'immagine tramite `mm.readImg` | Acquisizione dell'immagine nel *pipeline* di riconoscimento dei pattern |
| Codice LBP per pixel (EP07_04) | `local_binary_pattern(immagine, P=8, R=1, method="uniform")` |
| Istogramma di 10 contenitori per blocco | Funzione `descrittor_lbp` del Progetto Pratico 2 (`bins=10`, `range=(0, P+2)`) |
| Classificazione k-NN con metrica selezionabile (EP07_06) | `KNeighborsClassifier` addestrato su `X_textura` |
| Matrice di confusione $Ncl\times Ncl$ e accuratezza | `confusion_matrix` e `accuracy_score` su `yt_teste` |

Questo esercizio evidenzia, con pixel reali invece di valori sintetici, una limitazione discussa nella sezione finale del capitolo: classi di trama visivamente distinte per un osservatore umano — come **granulare** e **chiazzata** — possono produrre istogrammi LBP simili quando l'intorno considerato è piccolo ($R=1$), poiché entrambe presentano un'alta frequenza di pattern non uniformi alla scala di un singolo pixel. La classe **a scacchi**, invece, grazie ai suoi bordi regolari e ripetitivi, tende a essere separata con maggiore facilità. Ci si aspetta che la matrice di confusione prodotta rifletta esattamente questo pattern di confusione parziale.

#### 📦 Specifica di Input e Output (VPL)

**Input:**

```
L
C
[matrice L x C dell'immagine]
G S
Ncl nome_classe_1 ... nome_classe_Ncl
M k
N
nome_classe h0 h1 ... h9      (ripetuta N volte)
etichetta(0,0) etichetta(0,1) ... etichetta(G-1,G-1)
```

**Output:**

* $G \times G$ righe con la classe prevista di ciascun blocco, nell'ordine di lettura della griglia.
* $Ncl$ righe con la matrice di confusione (una riga per classe reale, valori separati da spazio).
* Ultima riga: `Acuracia: <valore>`.

#### 📌 Esempio (verifica manuale)

Per verificare l'implementazione del descrittore prima di testarla su un mosaico completo, si consideri un'immagine $6\times6$ **omogenea**, con tutti i pixel di intensità $100$, trattata come un unico blocco ($G=1$, $S=6$). Poiché ogni pixel interno ha gli 8 vicini con intensità uguale a quella del centro ($g_p \ge g_c$ in tutti i casi), tutti i bit $s_p$ valgono `1`, il numero di transizioni è $0$ (uniforme) e il contenitore è $\operatorname{popcount}(11111111)=8$. L'istogramma dell'unico blocco è, quindi, `0 0 0 0 0 0 0 0 1 0`.

| Input (riassunto) | Output | Osservazione |
|---|---|---|
| 6<br>6<br>[36 valori uguali a 100]<br>1 6<br>2 uniforme altra<br>euclidiana 1<br>2<br>uniforme 0 0 0 0 0 0 0 0 1 0<br>altra 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1<br>uniforme | uniforme<br>1 0<br>0 0<br>Acuracia: 1.0000 | La distanza del blocco al prototipo `uniforme` è esattamente $0$; la classe `altra` non appare nell'etichetta reale, quindi la sua riga nella matrice di confusione è nulla. |

#### 📌 File di Riferimento (.pgm)

Per il debug locale, due mosaici di test nel formato ASCII P2 sono messi a disposizione (allegati a questa consegna; integrandoli nel repository del capitolo, salvarli in `all/cap07/dati/EP07/`):

* 📥 **Caso 1 — Mosaico semplice (`Caso1_Mosaico_Simples.pgm`)**: griglia $2\times2$ di blocchi di $24\times24$ pixel, un campione di ciascuna delle quattro classi, con basso rumore — utile per validare la lettura dell'immagine e la logica di classificazione in uno scenario controllato.
* 📥 **Caso 2 — Mosaico misto (`Caso2_Mosaico_Misto.pgm`)**: griglia $3\times3$ di blocchi di $16\times16$ pixel, con classi ripetute e maggiore variabilità — scenario in cui la confusione tra **granulare** e **chiazzata** discussa nei Fondamenti Teorici tende a manifestarsi.

La [Figura 7.7](#fig-07-ep07) mostra i due mosaici, per un'ispezione visiva prima dell'implementazione.

In [ ]:
import os
import urllib.request
import numpy as np

def garantir_e_baixar_arquivo(nome_arquivo):
    diretorio_local = "dados/EP07"
    caminho_local = os.path.join(diretorio_local, nome_arquivo)
    
    # Creare la directory locale se non esiste
    if not os.path.exists(diretorio_local):
        os.makedirs(diretorio_local)
        
    # Se il file non esiste localmente, scaricalo dal repository remoto
    if not os.path.exists(caminho_local):
        url_base = "https://raw.githubusercontent.com/fzampirolli/"
        url_base += "pdi-vc/master/all/cap07/dados/EP07"
        url_arquivo = f"{url_base}/{nome_arquivo}"
        print(f"Scaricando {nome_arquivo} da GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
        except Exception as e:
            raise IOError(f"Erro ao baixar {nome_arquivo} do GitHub. ",
                          "Verifique a conexão ou a URL. Detalhes: {e}")
            
    return caminho_local

def ler_pgm_p2(caminho):
    with open(caminho) as f:
        linhas = [l for l in f.read().split() if l]
    assert linhas[0] == "P2"
    C, L = int(linhas[1]), int(linhas[2])
    maxv = int(linhas[3])
    valores = list(map(int, linhas[4:4 + L * C]))
    return np.array(valores, dtype=np.uint8).reshape(L, C)

# Garantisce il download e ottiene il percorso corretto
arq_caso1 = garantir_e_baixar_arquivo("Caso1_Mosaico_Simples.pgm")
arq_caso2 = garantir_e_baixar_arquivo("Caso2_Mosaico_Misto.pgm")

# Legge le matrici PGM
caso1 = ler_pgm_p2(arq_caso1)
caso2 = ler_pgm_p2(arq_caso2)

mm.show(
    [caso1, caso2],
    titles=[
        "Caso 1: Mosaico Semplice\n(blocchi 2x2, 1 campione/classe)",
        "Caso 2: Mosaico Misto\n(blocchi 3x3, classi ripetute)",
    ],
    cols=2,
    figsize=(8, 4),
)

**Figura 7.7:** Mosaici di riferimento (formato PGM ASCII) utilizzati nell


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0707" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">  
<style>
  #sim-ep0707 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0707 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #ede6d8; background: #f3efe6; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0707 button:hover { background: #e8e0cf; }
  #sim-ep0707 button.sim-ep0707_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0707_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #ede6d8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0707_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_07: Classificazione di Mosaico via LBP + k-NN</span>
  <span class="sim-ep0707_pill">⚫ pipeline completa</span>
</div>


  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
Mosaico 3x3 di blocchi 12x12 (L=C=36). LBP (P=8,R=1) calcolato pixel per pixel, con esclusione del bordo globale.      Regola k e la metrica e osserva la classificazione di ciascun blocco rispetto a 8 prototipi (2 per classe).
   
   </p>
     
<div style="background:#fff3cd;border:1px solid #ffe69c;border-radius:8px;padding:8px 12px;margin-bottom:12px;font-size:11px;color:#7a5c00;">
  ⚠️ Texture sintetiche generate da codice, non i file .pgm reali di EP07_07. Usa questo simulatore per capire il flusso dell'algoritmo, non come riferimento di difficoltà tra le classi.
</div>
     
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (numero di vicini)</label>
          <span id="ep0707_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
        </div>
<input id="ep0707_sl" style="width:100%;accent-color:#2980b9;" max="5" min="1" step="2" type="range" value="1">
      </div>
      <div>
        <label style="font-size:12px;font-weight:bold;color:#2980b9;display:block;margin-bottom:6px;">Metrica</label>
        <select id="ep0707_metric" style="font-size:12px;padding:4px 8px;border-radius:6px;border:1px solid #ccc;">
          <option value="euclidiana">euclidea</option>
          <option value="manhattan">manhattan</option>
        </select>
      </div>
    </div>

    <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;">
      <canvas id="ep0707_canvas" style="border-radius:8px;border:1px solid #ccc;"></canvas>
      <div id="ep0707_grid" style="flex:1;min-width:220px;display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <div id="ep0707_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;white-space:pre-line;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var S = 12, G = 3, L = G * S, SCALE = 5;   // bloco maior reduz o vazamento de borda; SCALE ajustado p/ manter o canvas ~180px
    var classesOrder = ["granular", "listrada", "manchada", "xadrez"];
    // Grade 3x3 com classes repetidas, análoga ao Caso 2 do enunciado
    var layout = [
      "granular", "listrada", "manchada",
      "xadrez",   "granular", "manchada",
      "listrada", "xadrez",   "granular"
    ];

    // --- Geração determinística de textura por pixel (didática, não os PGMs reais) ---
    function h(a, b, phase){
      var v = Math.sin((a + phase) * 12.9898 + (b + phase * 0.7) * 78.233 + phase * 3.1) * 43758.5453;
      return v - Math.floor(v);
    }
    function texturePixel(cls, r, c, phase){
      phase = phase || 0;
      switch(cls){
        case "granular": return h(r, c, phase) < 0.5 ? 220 : 30;
        case "listrada": return ((c + Math.floor(phase * 2)) % 4) < 2 ? 220 : 30;
        case "manchada": return h(Math.floor(r / 3), Math.floor(c / 3), phase) < 0.5 ? 200 : 60;
        case "xadrez":   return ((Math.floor(r / 2) + Math.floor(c / 2)) % 2 === 0) ? 230 : 20;
      }
    }

    function buildImage(){
      var img = [];
      for(var r = 0; r < L; r++){
        var row = [];
        for(var c = 0; c < L; c++){
          var bi = Math.floor(r / S), bj = Math.floor(c / S);
          row.push(texturePixel(layout[bi * G + bj], r, c, 0));
        }
        img.push(row);
      }
      return img;
    }

    // --- LBP: P=8, R=1, sentido horário, s_p = 1 se vizinho >= centro ---
    function lbpBin(patch, r, c){
      var center = patch[r][c];
      var neigh = [
        patch[r-1][c-1], patch[r-1][c], patch[r-1][c+1],
        patch[r][c+1],
        patch[r+1][c+1], patch[r+1][c], patch[r+1][c-1],
        patch[r][c-1]
      ];
      var bits = neigh.map(function(v){ return v >= center ? 1 : 0; });
      var trans = 0;
      for(var i = 0; i < 8; i++){ if(bits[i] !== bits[(i+1) % 8]) trans++; }
      if(trans <= 2) return bits.reduce(function(a,b){ return a+b; }, 0); // popcount 0..8
      return 9; // não uniforme
    }

    // Histograma de um patch isolado (usado para gerar protótipos), excluindo apenas a borda do patch
    function computeLBPHist(patch){
      var n = patch.length, m = patch[0].length;
      var hist = new Array(10).fill(0), count = 0;
      for(var r = 1; r < n - 1; r++){
        for(var c = 1; c < m - 1; c++){
          hist[lbpBin(patch, r, c)]++;
          count++;
        }
      }
      for(var k = 0; k < 10; k++) hist[k] = count > 0 ? hist[k] / count : 0;
      return hist;
    }

    // Histogramas por bloco da imagem completa, excluindo só a borda global (item 4/5 do enunciado)
    function computeMosaicHistograms(img){
      var hists = [], counts = [];
      for(var i = 0; i < G*G; i++){ hists.push(new Array(10).fill(0)); counts.push(0); }
      for(var r = 1; r < L - 1; r++){
        for(var c = 1; c < L - 1; c++){
          var bin = lbpBin(img, r, c);
          var idx = Math.floor(r/S) * G + Math.floor(c/S);
          hists[idx][bin]++;
          counts[idx]++;
        }
      }
      for(var b = 0; b < hists.length; b++){
        for(var k = 0; k < 10; k++) hists[b][k] = counts[b] > 0 ? hists[b][k] / counts[b] : 0;
      }
      return hists;
    }

    // --- Protótipos: 2 por classe (N=8), ordem de leitura fixa (usada no desempate) ---
    var prototypes = [];
    classesOrder.forEach(function(cls){
      [0, 5].forEach(function(phase){
        var Sp = S + 2, patch = [];
        for(var r = 0; r < Sp; r++){
          var row = [];
          for(var c = 0; c < Sp; c++) row.push(texturePixel(cls, r, c, phase));
          patch.push(row);
        }
        prototypes.push({ classe: cls, hist: computeLBPHist(patch) });
      });
    });

    function dist(u, v, metric){
      var s = 0;
      for(var i = 0; i < u.length; i++){
        s += metric === "euclidiana" ? (u[i]-v[i])*(u[i]-v[i]) : Math.abs(u[i]-v[i]);
      }
      return metric === "euclidiana" ? Math.sqrt(s) : s;
    }

    // Desempate de distância: ordem de leitura dos protótipos. Desempate de votação: ordem das classes.
    function classify(hist, k, metric){
      var cand = prototypes.map(function(p, idx){ return { classe: p.classe, d: dist(hist, p.hist, metric), idx: idx }; });
      cand.sort(function(a, b){ return a.d !== b.d ? a.d - b.d : a.idx - b.idx; });
      var viz = cand.slice(0, k);
      var votos = {};
      viz.forEach(function(v){ votos[v.classe] = (votos[v.classe] || 0) + 1; });
      var melhor = null, melhorN = -1;
      classesOrder.forEach(function(c){
        var n = votos[c] || 0;
        if(n > melhorN){ melhorN = n; melhor = c; }
      });
      return melhor;
    }

    var canvas = root.querySelector('#ep0707_canvas');
    canvas.width = L * SCALE; canvas.height = L * SCALE;
    var ctx = canvas.getContext('2d');
    var slK = root.querySelector('#ep0707_sl');
    var vlK = root.querySelector('#ep0707_vl');
    var selMetric = root.querySelector('#ep0707_metric');
    var gridEl = root.querySelector('#ep0707_grid');
    var dbg = root.querySelector('#ep0707_debug');

    var img = buildImage();
    var hists = computeMosaicHistograms(img);

    function render(){
      var k = parseInt(slK.value);
      var metric = selMetric.value;
      vlK.textContent = k;

      var preds = [];
      for(var idx = 0; idx < G*G; idx++) preds.push(classify(hists[idx], k, metric));

      var confusion = classesOrder.map(function(){ return new Array(classesOrder.length).fill(0); });
      var acertos = 0;
      for(var i2 = 0; i2 < G*G; i2++){
        var ri = classesOrder.indexOf(layout[i2]);
        var pi = classesOrder.indexOf(preds[i2]);
        confusion[ri][pi]++;
        if(layout[i2] === preds[i2]) acertos++;
      }
      var acc = acertos / (G*G);

      // Desenha a imagem real em tons de cinza
      for(var r = 0; r < L; r++){
        for(var c = 0; c < L; c++){
          var v = img[r][c];
          ctx.fillStyle = 'rgb(' + v + ',' + v + ',' + v + ')';
          ctx.fillRect(c*SCALE, r*SCALE, SCALE, SCALE);
        }
      }
      // Contorna cada bloco: verde = acerto, vermelho = erro
      for(var idx3 = 0; idx3 < G*G; idx3++){
        var bi = Math.floor(idx3 / G), bj = idx3 % G;
        ctx.strokeStyle = (preds[idx3] === layout[idx3]) ? '#10b981' : '#f43f5e';
        ctx.lineWidth = 2;
        ctx.strokeRect(bj*S*SCALE + 1, bi*S*SCALE + 1, S*SCALE - 2, S*SCALE - 2);
      }

      // Grade textual de apoio
      gridEl.innerHTML = '';
      for(var idx4 = 0; idx4 < G*G; idx4++){
        var ok = preds[idx4] === layout[idx4];
        var card = document.createElement('div');
        card.style.cssText = 'border-radius:8px;padding:6px;text-align:center;font-size:10px;border:2px solid ' + (ok ? '#10b981' : '#f43f5e') + ';';
        card.innerHTML = 'Real: ' + layout[idx4] + '<br><b style="color:' + (ok ? '#059669' : '#e11d48') + '">Pred: ' + preds[idx4] + (ok ? ' ✅' : ' ❌') + '</b>';
        gridEl.appendChild(card);
      }

      // Saída no mesmo formato do programa (itens 7-9 do enunciado)
      var linhas = [];
      linhas.push('Classes preditas (ordem de leitura da grade):');
      linhas.push(preds.join(' '));
      linhas.push('');
      linhas.push('Matriz de confusão (linhas=real, colunas=predita; ordem ' + classesOrder.join(',') + '):');
      confusion.forEach(function(lin){ linhas.push(lin.join(' ')); });
      linhas.push('');
      linhas.push('Acuracia: ' + acc.toFixed(4));
      dbg.textContent = linhas.join('\\n');
    }

    slK.addEventListener('input', render);
    selMetric.addEventListener('change', render);
    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0707');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.8:** Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN


<figure id="fig-07-sim-ep0707">
  <img src="imagens/fig-07-sim-ep0707.png" alt=" Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN " style="max-width:80%" />
  <figcaption><strong>Figura 7.8:</strong>  Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN </figcaption>
</figure>

In [ ]:
%%writefile EP07_07.py
# Codice Python

In [ ]:
TestSuite("EP07_07.py").run()